In [1]:
# This notebook reads logic rules directly from the inspectable rule system that
# now ships with the OWR codebase (source/logic/AccessRule.py); no manual source
# edits are required anymore.
#
# Every Entrance/Location exposes:
#   spot.verbose_rule -- a structured AccessRule tree (inspectable, combinable)
#   spot.access_rule  -- the compiled callable used for logic evaluation
#
# The helper cell added right after the format_rule / mapping definitions converts
# verbose_rule into the legacy "|token|" text via rule_to_text(...) and stores it on
# spot.text_access_rule, so the rest of this notebook is unchanged.
#
# Legacy lambda leaves render via dill (Opaque.__str__ in AccessRule.py); if dill is
# unavailable the leaf degrades to "Opaque(<lambda>)" rather than crashing.
#
# (Previously this required hand-editing BaseClasses.py and Rules.py to add
# text_access_rule / clean_text_rule. That is no longer needed.)

In [2]:
from BaseClasses import World

In [3]:
from CLI import parse_cli, parse_settings
from Dungeons import dungeon_keys, dungeon_bigs
from Main import main
from OWEdges import OWTileRegions

from source.classes.BabelFish import BabelFish

args = parse_settings()
args = parse_cli({})

lang = "en"
fish = BabelFish(lang=lang)


In [4]:
args.pottery[1] = 'lottery'
args.dropshuffle[1] = 'underworld'
args.bonk_drops[1] = True

args.logic[1] = 'noglitches'
world: World = main(args=args, fish=fish, seed=0)
worldAllState = world.get_all_state(keys=False)

In [5]:
args.pottery[1] = 'lottery'
args.dropshuffle[1] = 'underworld'
args.bonk_drops[1] = True
args.logic[1] = 'noglitches'
args.mode[1] = 'inverted'
invertedWorld: World = main(args=args, fish=fish)
invertedAllState = invertedWorld.get_all_state(keys=False)

In [6]:
from BaseClasses import CrystalBarrier

hasCrystalSwitch = set()
blueBarrier = set()
orangeBarrier = set()



for d in world.doors:
    if d.crystal == CrystalBarrier.Either:
        hasCrystalSwitch.add(d.name)
    elif d.crystal == CrystalBarrier.Blue:
        blueBarrier.add(d.name)
    elif d.crystal == CrystalBarrier.Orange:
        orangeBarrier.add(d.name)

In [7]:
import ast

In [8]:
def find_no_exit_doors(file_path):
    with open(file_path, "r") as f:
        tree = ast.parse(f.read())

    no_exit_doors = []

    class DoorVisitor(ast.NodeVisitor):
        def visit_Call(self, node):
            # Check if this call or any parent call in the chain is .no_exit()
            # and if the root is create_door
            
            # We want to find the whole chain
            if self.is_no_exit_call(node):
                door_name = self.get_create_door_name(node)
                if door_name:
                    no_exit_doors.append(door_name)
            
            self.generic_visit(node)

        def is_no_exit_call(self, node):
            # Check if .no_exit is anywhere in the call chain of node
            curr = node
            while isinstance(curr, ast.Call):
                if isinstance(curr.func, ast.Attribute):
                    if curr.func.attr == 'no_exit':
                        return True
                    curr = curr.func.value
                else:
                    break
            return False

        def get_create_door_name(self, node):
            # Find the create_door call at the base of the chain and return the door name
            curr = node
            while isinstance(curr, ast.Call):
                if isinstance(curr.func, ast.Name) and curr.func.id == 'create_door':
                    # First argument is player, second is door name
                    if len(curr.args) >= 2 and isinstance(curr.args[1], ast.Constant):
                        return curr.args[1].value
                    elif len(curr.args) >= 2 and isinstance(curr.args[1], ast.Str): # for older python
                        return curr.args[1].s
                    return None
                
                if isinstance(curr.func, ast.Attribute):
                    curr = curr.func.value
                else:
                    break
            return None

    visitor = DoorVisitor()
    visitor.visit(tree)
    return sorted(list(set(no_exit_doors)))

one_way_doors = find_no_exit_doors("Doors.py")


In [9]:
format_test_cases = [
    ("|False)  |", {"always": {"allOf": ["never"]}}),
    ("", {}),
    (
        "((|can_use_bombs|) and (|has_Pearl|)) and (|has_Pearl|)",
        {"always": {"allOf": ["bomb", "moonpearl"]}},
    ),
    ("|Hookshot|", {"always": {"allOf": ["hookshot"]}}),
    (
        "|can_shoot_arrows or Red Boomerang or Fire Rod or Ice Rod or Cane of Somaria) |",
        {"always": {"anyOf": ["bow", "boomerang", "firerod", "icerod", "somaria"]}},
    ),
    (
        "((|can_reach_blue(world.get_region(Mire Crystal Dead End|) or (|eval_alternative_crystal_main( door_name, dungeon|)) and (|has_Pearl|)",
        {"always": {"allOf": ["moonpearl"]}},
    ),
    (
        "(|can_reach_blue(world.get_region(TR Crystal Maze Start|) and (|has_Pearl|)",
        {"always": {"allOf": ["moonpearl"]}},
    ),
    (
        "|Mushroom and can_reach(Potion Shop Area, Region|",
        {"always": {"allOf": ["mushroom", "canReach|Potion Shop Area"]}},
    ),
    (
        "| | and |moonpearl|",
        {"always": {"allOf": ["moonpearl"]}},
    ),
    (
        "|Lamp or world.free_lamp_cone[player]|",
        {"always": {"allOf": ["lantern"]}},
    ),
    (
        "(|eval_small_key_door_partial_main( door_name, dungeon|) and (|Lamp or world.free_lamp_cone[player]|)",
        {"always": {"allOf": ["lantern"]}},
    )
]

enemy_format_test_cases = [
    (
        "(|rule1( or rule2(|) and (|has_Pearl|)",
        {"always": {"allOf": ["melee", "bomb", "moonpearl"]}},
        "StalfosKnight",
    ),
    ("|has(item, player, count|", {"always": {"allOf": ["hammer"]}}, "Terrorpin"),
]

mapping = {
    "can_use_bombs": "bomb",
    "can_shoot_arrows": "bow",
    "has_Pearl": "",
    "Red Boomerang": "boomerang",
    "Blue Boomerang": "boomerang",
    "Book of Mudora": "book",
    "Fire Rod": "firerod",
    "Ice Rod": "icerod",
    "Magic Powder": "powder",
    "Cane of Somaria": "somaria",
    "Cane of Byrna": "byrna",
    "Ocarina": "flute",
    "Lamp": "lantern",
    "green pendant": "greenPendant",
    "red pendant": "canPullPedestal",
    "blue pendant": "canPullPedestal",
    "has_sword": "sword",
    "tempered sword": "temperedSword",
    "golden sword": "goldSword",
    "has_boots": "boots",
    "has_mirror": "mirror",
    "Pegasus Boots": "boots",
    "has_bottle": "bottle",
    "silver arrows": "canUseSilverArrows",
    "escort old man": "canCollectOldMan",
    "return old man": "canRescueOldMan",
    "pick up kiki": "canCollectKiki",
    "dark palace opened": "canOpenPod",
    "get frog": "canCollectFrog",
    "return smith": "canRescueBlacksmith",
    "pick up purple chest": "canCollectPurpleChest",
    "deliver purple chest": "canDeliverPurpleChest",
    "pick up big bomb": "canCollectBigBomb",
    "detonate big bomb": "canOpenPyramidFairy",
    "crystal 5": "canBuyBigBomb",
    "crystal 6": "canBuyBigBomb",
    "world.is_pyramid_open": "isPyramidOpen",
    "can_kill_most_things": "canKillMostEnemies",
    "has_misery_mire_medallion": "medallion|mm",
    "has_turtle_rock_medallion": "medallion|tr",
    "turtle opened": "canOpenTR",
    "beat agahnim 2": "agahnim2",
    "can_reach_orange.*": "", #"orangeSwitchDown",
    "can_reach_blue.*": "", #"blueSwitchDown",
    "can_reach\((.*),.* ": r"canReach|\1",
    "can_flute": "canFlute",
    "can_extend_magic\(, (\d+).*": r"canExtendMagic|\1",
    "can_hit_crystal": "canHitSwitch",
    "can_hit_crystal_through_barrier": "canHitRangedSwitch",
    "can_lift_rocks": "glove",
    "can_lift_heavy_rocks": "mitts",
    "has_fire_source": "canLightFires",
    "can_collect_bonkdrops": "canGetBonkableItem",
    "can_avoid_lasers": "canAvoidLasers",
    "can_melt_things": "canBurnThings",
    "has_beaten_aga": "agahnim",
    "world.free_lamp_cone.*": "",
    "has_beam_sword": "swordbeams",
    "def hidden_pits_rule: return hidden pits": "",
    "entrance.parent_region.dungeon.*": "",
    "world.can_take_damage": "canTakeDamage",
    "has(item_name": "",
    "not trench 1 filled": "",
    "trench 1 filled": "",
    "not trench 2 filled": "",
    "trench 2 filled": "",
    "drained swamp": "",
    "has_crystals(world.crystals_needed_for_gt[player]": "canOpenGT",
    "has_crystals(world.crystals_needed_for_gt[]": "canOpenGT",
    "has_crystals(world.crystals_needed_for_ganon[]": "canBeatGanon",
    "set_rule(world.get_location(spike cave, lambda  hammer": "",  # Spike cave hammer is only picked up on certain things
    "not has_beaten_aga": "",
    "true": "",
    "rule1": "",
    "rule2": "",
    "item": "",
    "world.logic[] in [noglitches, minorglitches]": "",
    "has_hearts(.*": "",
    "open floodgate": "canReach|Dam",
    "shining light": "canOpenTTAttic",
    "maiden unmasked, entrance.": "canRevealBlind",
    "maiden rescued": "canRevealBlind",
    "world.get_region.*": "",
    "convenient block": "canReach|Ice Crystal Block",
    "def hidden_pits_rule(: return hidden pits": "",
    "hidden_pits_rule": "",
    "has_sm_key.*": "",
    "has_hearts.*": "",
    "agahnim, or": "",
    "boolean": "",
    "eval_small_key_door.*": "",
    "eval_alternative_crystal.*": "",
}

ow_mapping = mapping.copy()
ow_mapping.update(
    {
        "has_Pearl": "moonpearl",
    }
)

enemy_rules = {
    "Pokey": "canBurnThings",
    "RedBari": "canBurnThings",
    "RedEyegore": "bow",
    "Terrorpin": "hammer",
    "StalfosKnight": "melee and bomb",
}

import ast
import re


def format_rule(rule, mapping=mapping, isEnemy=""):
    """ "
    Rules are output as combinations of allOf and anyOf objects, with requirements being in lists.
    If not possible, return "never"
    If no rules, return {} (i.e. always true)
    Rules are always encapsulated in in a higher level object "always"
    i.e.:
        {
            "always": {
                "allOf": [
                    {
                        "anyOf": [
                            "pearl",
                            "flute"
                        ]
                    },
                    "hookshot"
                ]
            }
        }

    There are many edge cases to consider:
    - Some requirements will be empty strings - these should return as true (i.e. {})
    - All requirements are extracted extracted from lambda functions should be encapsulated in pipes `|requirement|`
    - Requirements that used the built in OR/AND functions have been encapsulated in brackets `(|requirement1| or |requirement2|)`/`(|requirement1| and |requirement2|)`
    - Some requirements have an " or " or " and " statement within them from the lambda text rules, these will _not_ be further encapsulated in parentheses because they come from the same rule
    - Some lambda text rules will have python syntax in them, such as:
      - `|has_crystals(world.crystals_needed_for_gt[player]|`
      - `world.is_pyramid_open`
      - `|has_crystals(world.crystals_needed_for_gt[player] or world.shuffle[player] in (restricted, full, lite, lean, district, swapped, crossed, insanity|`
    - Some requirements will have multiple words separated by spaces, such as:
      - `|Turtle Opened|`
      - `|world.can_take_damage or Cape or Cane of Byrna or has_sword|`
        - This is actually a larger or statement that should result in the following `(|world.can_take_damage| or |Cape| or |Cane of Byrna| or |has_sword|)` and then be converted to JSON appropriately
    - Some requirements are duplicated, e.g. `((|can_lift_rocks|) and (|has_Pearl|)) and (|has_Pearl|)`. It would be better to deduplicate these to `(|can_lift_rocks|) and (|has_Pearl|)`
    - Some requirements may not be determinable and will say `|boolean|`. Some other rules could not be extracted properly and will say `|rule1( or rule2(|`.
      - These should be able to be changed using an old -> new mapping provided externally
    - Some requirements may have data in them that is not the same each time, and we might need to use regex to extract the core requirement, i.e.
      - `"|can_reach_orange(world.get_region(PoD Arena North|"` -> `"|canHitSwitch|"`
    - In some cases, the isEnemy parameter is provided, and if the rule matches an enemy in the enemy_rules mapping, it should add the rule with that requirement.
      - Usually the python-esque code should be replaced with the enemy requirement as well.
    """
    if not rule:
        return {}

    # Pre-cleaning
    rule = rule.strip()

    # Create lowercased mapping, stripping keys
    lower_mapping = {k.lower().strip(): v for k, v in mapping.items()}

    # 1. Replace |...| with placeholders
    placeholders = {}
    counter = 0

    def sub_cb(match):
        nonlocal counter
        key = f"REQ_{counter}"
        placeholders[key] = match.group(1)
        counter += 1
        return key

    processed_rule = re.sub(r"\|([^|]+)\|", sub_cb, rule)

    # If processed_rule is empty or just whitespace, return {}
    if not processed_rule.strip():
        return {}

    # 2. Parse with AST
    try:
        tree = ast.parse(processed_rule, mode="eval")
    except SyntaxError:
        # Fallback for simple strings or malformed
        if "False" in rule:
            return {"always": {"allOf": ["never"]}}
        return {"always": {"allOf": ["never"]}}

    # 3. Recursive processor
    def parse_node(node):
        if isinstance(node, ast.BoolOp):
            op = "allOf" if isinstance(node.op, ast.And) else "anyOf"
            values = [parse_node(v) for v in node.values]
            return {op: values}
        elif isinstance(node, ast.Name):
            content = placeholders.get(node.id, "")
            return process_requirement(content)
        elif isinstance(node, ast.Expression):
            return parse_node(node.body)
        return {}

    def process_requirement(req_str):
        # Clean up
        req_str = req_str.strip()

        # Handle has(...)
        # has('Item')
        m = re.search(r"has\(['\"]([^'\"]+)['\"]\)", req_str)
        if m:
            req_str = m.group(1)
        # has(item, ...)
        m = re.search(r"has\(([^,]+),", req_str)
        if m:
            req_str = m.group(1)

        # Remove `lambda state:`, `state.`, `(player)`, `player`
        req_str = req_str.replace("lambda state:", "")
        req_str = req_str.replace("state.", "")
        req_str = req_str.replace("(player)", "")
        req_str = req_str.replace("player", "")

        # Remove trailing/leading parens/spaces
        req_str = req_str.strip("() ")

        # Do NOT lowercase here to preserve case for regex replacements

        return parse_simple_rule(req_str)

    def apply_mapping(req_str):
        # Handle isEnemy
        if isEnemy and isEnemy in enemy_rules:
            # Heuristic: if req_str contains "rule" or "boolean" or "item", replace with enemy rule
            # Check lowercased for heuristic
            lower_req = req_str.lower()
            if "rule" in lower_req or "boolean" in lower_req or "item" in lower_req:
                return parse_simple_rule(enemy_rules[isEnemy])

        # Apply mapping
        # First pass: exact match (case insensitive)
        if req_str.lower() in lower_mapping:
            res = lower_mapping[req_str.lower()]
            if res == "":
                return ""  # Return empty string to be filtered out, not {} which is "always true"
            return res

        # Second pass: partial replacement and regex
        sorted_keys = sorted(lower_mapping.keys(), key=len, reverse=True)
        replaced = False
        for k in sorted_keys:
            v = lower_mapping[k]
            # Try literal first (case insensitive)
            if k in req_str.lower():
                # Use re.sub with IGNORECASE to replace literal k
                # Escape k to treat it as literal
                try:
                    new_str = re.sub(re.escape(k), v, req_str, flags=re.IGNORECASE)
                    if new_str != req_str:
                        req_str = new_str
                        replaced = True
                except re.error:
                    pass
                continue

            # Try regex if it looks like one (contains *)
            if "*" in k or "\\" in k or "(" in k:
                try:
                    new_str = re.sub(k, v, req_str, flags=re.IGNORECASE)
                    if new_str != req_str:
                        req_str = new_str
                        replaced = True
                except re.error:
                    pass

        # Check if result is empty after replacements
        req_str = req_str.strip()
        if not req_str:
            return ""  # Return empty string to be filtered out in anyOf, not {} which short-circuits

        # Handle eval_ heuristic - treat as ignored since these are
        # dynamic checks that we handle elsewhere (like small key doors)
        if req_str.lower().startswith("eval_"):
            return ""

        # Handle "False"
        if "false" in req_str.lower():
            return "never"

        # If no replacement happened, lowercase it (default behavior)
        if not replaced:
            return req_str.lower()

        return req_str

    def parse_simple_rule(text):
        if " or " in text:
            parts = [parse_simple_rule(p.strip()) for p in text.split(" or ")]
            return {"anyOf": parts}
        if " and " in text:
            parts = [parse_simple_rule(p.strip()) for p in text.split(" and ")]
            return {"allOf": parts}

        text = text.strip()
        # Use strip instead of replace to preserve internal parens
        text = text.strip("() ")

        return apply_mapping(text)

    result = parse_node(tree)

    # 4. Simplify
    def simplify(obj):
        # if obj == "": return {} # Treat empty string as empty dict (True)

        if isinstance(obj, dict):
            if "allOf" in obj:
                items = [simplify(x) for x in obj["allOf"]]
                # Flatten
                new_items = []
                for i in items:
                    if isinstance(i, dict) and "allOf" in i:
                        new_items.extend(i["allOf"])
                    else:
                        new_items.append(i)
                # Filter empty
                new_items = [i for i in new_items if i != {} and i != ""]
                # Deduplicate
                unique = []
                for i in new_items:
                    if i not in unique:
                        unique.append(i)

                if "never" in unique:
                    return "never"
                if not unique:
                    return {}
                if len(unique) == 1:
                    return unique[0]
                return {"allOf": unique}

            if "anyOf" in obj:
                items = [simplify(x) for x in obj["anyOf"]]
                new_items = []
                for i in items:
                    if isinstance(i, dict) and "anyOf" in i:
                        new_items.extend(i["anyOf"])
                    else:
                        new_items.append(i)

                # Check for True ({})
                if any(i == {} for i in new_items):
                    return {}

                # Filter "" but track if we saw it
                has_empty_str = "" in new_items
                new_items = [i for i in new_items if i != ""]

                unique = []
                for i in new_items:
                    if i not in unique:
                        unique.append(i)

                if "never" in unique:
                    unique.remove("never")

                if not unique:
                    return "" if has_empty_str else "never"

                if len(unique) == 1:
                    return unique[0]
                return {"anyOf": unique}
        return obj

    final = simplify(result)

    # Special handling for enemy rules to match test expectation
    if isEnemy and isinstance(final, str) and final != "never" and final != "":
        final = {"allOf": [final]}

    # Ensure single items are wrapped in allOf
    if isinstance(final, str) and final != "":
        final = {"allOf": [final]}

    if final == "":
        return {}

    if not final:
        return {}

    return {"always": final}


# Run tests
print("Running tests...")
for rule, expected in format_test_cases:
    result = format_rule(rule, ow_mapping)
    if result != expected:
        print(f"FAILED: {rule}\nExpected: {expected}\nGot: {result}")
    else:
        print(f"PASSED: {rule} -> {result}")

for rule, expected, enemy in enemy_format_test_cases:
    result = format_rule(rule, ow_mapping, enemy)
    if result != expected:
        print(f"FAILED: {rule} (Enemy: {enemy})\nExpected: {expected}\nGot: {result}")
    else:
        print(f"PASSED: {rule} (Enemy: {enemy}) -> {result}")


def check_rule_keys(loc, state=worldAllState):
    # We need to check if keys are needed, this is easy for single entrance dungeons
    if loc.parent_region.dungeon:
        dungeon_name = loc.parent_region.dungeon.name
        sk = 0
        bk = 0
        for i in range(0, 12):
            stateCopy = state.copy()
            r = False
            stateCopy.prog_items[dungeon_keys[dungeon_name], 1] += 1
            for bk in range(0, 2):
                stateCopy.prog_items[dungeon_bigs[dungeon_name], 1] += 1
                if loc.access_rule(state):
                    r = True
                    if sk > 0 or bk > 0:
                        print(
                            "Location",
                            loc.name,
                            "in",
                            loc.parent_region.name,
                            "needs",
                            sk,
                            "small keys and",
                            bk,
                            "big keys",
                        )
                    break
            if r:
                if sk > 0 or bk > 0:
                    print(
                        "Location",
                        loc.name,
                        "in",
                        loc.parent_region.name,
                        "needs",
                        sk,
                        "small keys and",
                        bk,
                        "big keys",
                    )
                break
        # if not r:
        # print("Location", loc.name, "in", loc.parent_region.name, "needs more than 11 small keys and 1 big key")
    return format_rule(loc.text_access_rule, ow_mapping if loc.parent_region.type in [RegionType.LightWorld, RegionType.DarkWorld] else mapping)



Running tests...
PASSED: |False)  | -> {'always': {'allOf': ['never']}}
PASSED:  -> {}
PASSED: ((|can_use_bombs|) and (|has_Pearl|)) and (|has_Pearl|) -> {'always': {'allOf': ['bomb', 'moonpearl']}}
PASSED: |Hookshot| -> {'always': {'allOf': ['hookshot']}}
PASSED: |can_shoot_arrows or Red Boomerang or Fire Rod or Ice Rod or Cane of Somaria) | -> {'always': {'anyOf': ['bow', 'boomerang', 'firerod', 'icerod', 'somaria']}}
PASSED: ((|can_reach_blue(world.get_region(Mire Crystal Dead End|) or (|eval_alternative_crystal_main( door_name, dungeon|)) and (|has_Pearl|) -> {'always': {'allOf': ['moonpearl']}}
PASSED: (|can_reach_blue(world.get_region(TR Crystal Maze Start|) and (|has_Pearl|) -> {'always': {'allOf': ['moonpearl']}}
PASSED: |Mushroom and can_reach(Potion Shop Area, Region| -> {'always': {'allOf': ['mushroom', 'canReach|Potion Shop Area']}}
PASSED: | | and |moonpearl| -> {'always': {'allOf': ['moonpearl']}}
PASSED: |Lamp or world.free_lamp_cone[player]| -> {'always': {'allOf': ['la

In [10]:
# --- Bridge the structured AccessRule system to the legacy text pipeline ---
# Renders each spot.verbose_rule (see source/logic/AccessRule.py) back into the
# "|token|" grammar that format_rule / mapping expect, then stores it on
# spot.text_access_rule so the downstream cells work unchanged.
from source.logic.AccessRule import (
    AndRule, OrRule, NotRule, Has, Primitive, Reach, Opaque, TrueRule, FalseRule,
)

# A few structured leaf names differ from the old lambda-derived tokens.
for _m in (mapping, ow_mapping):
    _m.setdefault("Magic Mirror", "mirror")
    _m.setdefault(r"can_extend_magic\(\d+, (\d+).*", r"canExtendMagic|\1")
    # OWG techniques -> tracker sequence-break tokens (see requirementEvaluator.ts)
    _m["can_boots_clip_lw"] = "canBootsClip"
    _m["can_boots_clip_dw"] = "canBootsClip"
    _m["can_bunny_pocket"] = "canBunnyPocket"
    # HMG clips (OWR can_dash_clip / can_bomb_clip take the clip region; the tracker uses the link state there)
    _m["can_dash_clip"] = "canDashClip"
    _m["can_bomb_clip"] = "canBombClip"
    # Waterwalk is not in OWR's logic; the glitched variant worlds add it (see build_variant_world)
    _m["can_water_walk"] = "canWaterWalk"
    # Another dungeon's keys (HMG: Mire keys in Swamp, Mire/Hera big keys in Swamp/Hera). process_requirement
    # strips the closing paren, hence the trailing ".*"
    _m[r"xdungeon_bigkey\((\w+).*"] = r"otherBigKey|\1"
    _m[r"xdungeon_smallkeys\((\w+), (\d+).*"] = r"otherSmallKeys|\1|\2"
    _m["has_blunt_weapon"] = "has_blunt_weapon"  # tracker treats this as always-available
    # Structured conversion expands has_sword()/has_boomerang() helpers into
    # individual items; fold the base ones back to canonical combat tokens so the
    # output matches the tracker vocabulary (progression swords keep their own
    # temperedSword/goldSword tokens; higher swords already imply "sword").
    _m["fighter sword"] = "sword"          # sword count >= 1 (any sword)
    _m["master sword"] = "swordbeams"      # sword count >= 2 (beam-capable)
    _m["red_boomerang"] = "boomerang"
    # has_crystals resolves to an int in structured rules, so GT vs Ganon can only
    # be told apart by which spot the rule sits on (see populate_text_rules below).
    _m["has_crystals_gt"] = "canOpenGT"
    _m["has_crystals_ganon"] = "canBeatGanon"
mapping.setdefault("Moon Pearl", "")
ow_mapping["Moon Pearl"] = "moonpearl"

# Spots whose has_crystals requirement gates Ganon's Tower; anything else that
# carries has_crystals (i.e. the Ganon fight) gates beating Ganon.
GT_CRYSTAL_SPOTS = {"Ganons Tower", "Agahnims Tower", "GT Approach", "GT Leave"}


_XDUNGEON_IDS = {
    'Hyrule Castle': 'hc', 'Eastern Palace': 'ep', 'Desert Palace': 'dp', 'Tower of Hera': 'toh',
    'Agahnims Tower': 'ct', 'Palace of Darkness': 'pod', 'Swamp Palace': 'sp', 'Skull Woods': 'sw',
    'Thieves Town': 'tt', 'Ice Palace': 'ip', 'Misery Mire': 'mm', 'Turtle Rock': 'tr', 'Ganons Tower': 'gt',
}
_SMALL_KEY_DUNGEON = {v: k for k, v in dungeon_keys.items()}
_BIG_KEY_DUNGEON = {v: k for k, v in dungeon_bigs.items()}


def rule_to_text(rule, crystals_token=None, dungeon=None):
    """Serialize a verbose_rule AccessRule tree into the legacy |token| text."""
    if rule is None or isinstance(rule, TrueRule):
        return ""
    if isinstance(rule, FalseRule):
        return "|False|"
    if isinstance(rule, AndRule):
        return "(" + " and ".join(rule_to_text(r, crystals_token, dungeon) for r in rule.rules) + ")"
    if isinstance(rule, OrRule):
        return "(" + " or ".join(rule_to_text(r, crystals_token, dungeon) for r in rule.rules) + ")"
    if isinstance(rule, NotRule):
        return f"|not {rule_to_text(rule.rule, crystals_token, dungeon).strip('|() ')}|"
    if isinstance(rule, Has):
        # Dungeon keys resolve to the tracker's context-aware bigkey/smallkey
        # tokens (the dungeon comes from the region, not the token); these are
        # also skipped by the owg|/hmg| prefixing, which is correct.
        # Another dungeon's key (HMG clips) gets its own token so the tracker doesn't
        # treat the door as one of this dungeon's key doors.
        if rule.item.startswith("Big Key ("):
            owner = _BIG_KEY_DUNGEON.get(rule.item)
            if dungeon and owner and owner != dungeon:
                return f"|xdungeon_bigkey({_XDUNGEON_IDS[owner]})|"
            return "|bigkey|"
        if rule.item.startswith("Small Key ("):
            owner = _SMALL_KEY_DUNGEON.get(rule.item)
            if dungeon and owner and owner != dungeon:
                return f"|xdungeon_smallkeys({_XDUNGEON_IDS[owner]}, {rule.count})|"
            return "|smallkey|"
        return f"|{rule.item}|"
    if isinstance(rule, Primitive):
        if rule.method_name == "has_crystals" and crystals_token:
            return f"|{crystals_token}|"
        # Old clean_text produced bare method names (the arg was the player); keep
        # that, but preserve can_extend_magic's amount which the mapping consumes.
        if rule.method_name == "can_extend_magic":
            return f"|{rule.method_name}({', '.join(str(a) for a in rule.args)})|"
        return f"|{rule.method_name}|"
    if isinstance(rule, Reach):
        # Matches mapping regex `can_reach\((.*),.* ` -> canReach|<spot>
        return f"|can_reach({rule.spot}, Region |"
    if isinstance(rule, Opaque):
        # Opaque.__str__ yields |cleaned lambda| via dill when source is available.
        s = str(rule)
        return s if s.startswith("|") else f"|{s}|"
    s = str(rule)
    return s if (s.startswith("(") or "|" in s) else f"|{s}|"


def _crystals_token_for(spot):
    if spot.name in GT_CRYSTAL_SPOTS:
        return "has_crystals_gt"
    if spot.name == "Ganon":
        return "has_crystals_ganon"
    return None


def populate_text_rules(target_world):
    """Fill spot.text_access_rule from spot.verbose_rule for all exits/locations."""
    for region in target_world.regions:
        for spot in list(region.exits) + list(region.locations):
            vr = getattr(spot, "verbose_rule", None)
            dungeon = region.dungeon.name if region.dungeon else None
            spot.text_access_rule = rule_to_text(vr, _crystals_token_for(spot), dungeon) if vr is not None else ""


populate_text_rules(world)
populate_text_rules(invertedWorld)
# If the OWG/HMG variants in the process_world_variant cell are re-enabled,
# call populate_text_rules(...) on those worlds too before processing them.

In [11]:
ignored_tos = [
    "Flute Sky",
]

import json
from BaseClasses import RegionType

all_rules = set()

entrance_graph = {}

all_doors = {d.name: d for d in world.doors}
all_doors_inverted = {d.name: d for d in invertedWorld.doors}

# , "Inverted": invertedWorld
for region in world.regions:
    entrance_graph[region.name] = {
        "exits": {},
        "entrances": [],
        "type": region.type.name,
        "locations": {},
    }
    exits = entrance_graph[region.name]["exits"]
    entrances = entrance_graph[region.name]["entrances"]
    locations = entrance_graph[region.name]["locations"]
    reg = entrance_graph[region.name]
    if region.type in [RegionType.LightWorld, RegionType.DarkWorld]:
        reg['owid'] = OWTileRegions.get(region.name, None)
    else:
        for x in region.exits:
            door = all_doors.get(x.name, None)
            if door:
                reg['tileid'] = door.roomIndex
    for exit in region.exits:
        # Track entrances from OW to UW
        if exit.connected_region:
            if exit.name.startswith("Mirror From ") or exit.name.startswith(
                "Mirror To "
            ):
                # Add "has_mirror" requirement
                exit.text_access_rule = (
                    f"(|has_mirror|) and ({exit.text_access_rule})"
                    if exit.text_access_rule
                    else "|has_mirror|"
                )

            if exit.connected_region.name in ignored_tos:
                continue
            if exit.connected_region.type in [
                RegionType.Cave,
                RegionType.Dungeon,
            ] and region.type in [RegionType.LightWorld, RegionType.DarkWorld]:
                entrances.append(exit.name)
            exit_dict = {
                "to": exit.connected_region.name,
                "type": exit.connected_region.type.name,
                "requirements": {"Open": check_rule_keys(exit), "Inverted": "never"},
            }
            exits[exit.name] = exit_dict
            all_rules.add(json.dumps(check_rule_keys(exit)))
    for location in region.locations:
        locations[location.name] = {
            "requirements": {"Open": check_rule_keys(location), "Inverted": "never"}
        }
        all_rules.add(json.dumps(check_rule_keys(location)))
        if "Enemy" in location.name:
            locations[location.name]["type"] = location.note

In [12]:
for region in invertedWorld.regions:
    exits = entrance_graph[region.name]["exits"]
    entrances = entrance_graph[region.name]["entrances"]
    locations = entrance_graph[region.name]["locations"]
    reg = entrance_graph[region.name]
    if region.type in [RegionType.LightWorld, RegionType.DarkWorld]:
        reg['owid'] = OWTileRegions.get(region.name, None)
    else:
        for x in region.exits:
            door = all_doors_inverted.get(x.name, None)
            if door:
                reg['tileid'] = door.roomIndex
    for exit in region.exits:
        if exit.name.startswith("Mirror From ") or exit.name.startswith("Mirror To "):
            exit.text_access_rule = (
                f"(|has_mirror|) and ({exit.text_access_rule})"
                if exit.text_access_rule
                else "|has_mirror|"
            )
        # Track entrances from OW to UW
        if exit.connected_region:
            if exit.connected_region.name in ignored_tos:
                continue
            exit_dict = {
                "to": exit.connected_region.name,
                "type": exit.connected_region.type.name,
                "requirements": {"Open": "never", "Inverted": check_rule_keys(exit)},
            }
            all_rules.add(json.dumps(check_rule_keys(exit)))
            # Inverted only region
            if exit.name not in exits:
                if region.type in [RegionType.LightWorld, RegionType.DarkWorld]:
                    entrances.append(exit.name)
                    exits[exit.name] = exit_dict
            # Inverted has unique connected region from same exit
            elif exits[exit.name]["to"] != exit.connected_region.name:
                exits[exit.name + " (Inverted)"] = exit_dict
            else:
                exit_dict = exits[exit.name]
                exit_dict["requirements"]["Inverted"] = check_rule_keys(exit)
                all_rules.add(json.dumps(check_rule_keys(exit)))
    for location in region.locations:
        if location.name not in locations:
            locations[location.name] = {
                "requirements": {"Open": "never", "Inverted": check_rule_keys(location)}
            }
            all_rules.add(json.dumps(check_rule_keys(location)))
            if "Enemy" in location.name:
                locations[location.name]["type"] = location.note
        else:
            locations[location.name]["requirements"]["Inverted"] = check_rule_keys(
                location
            )
            all_rules.add(json.dumps(check_rule_keys(location)))

In [13]:
def prefix_rules(rule, prefix="hmg|"):
    """Recursively prefix all rule tokens in a requirement structure with the given prefix.
    Leaves 'never' and empty strings unchanged. An unconditional rule ({}) has no tokens
    to prefix, so it becomes an explicit "<prefix>true" to stay gated on the glitch mode."""
    if rule == {}:
        return {"always": {"allOf": [f"{prefix}true"]}}
    if isinstance(rule, str):
        if rule in ("never", "", "bigkey", "smallkey"):
            return rule
        if rule.startswith("canReach|"):
            return f"{prefix}{rule}"
        return f"{prefix}{rule}"
    elif isinstance(rule, dict):
        result = {}
        for k, v in rule.items():
            if v == {}:
                result[k] = {"allOf": [f"{prefix}true"]}
            elif isinstance(v, list):
                result[k] = [prefix_part(item, prefix) for item in v]
            elif isinstance(v, dict):
                result[k] = prefix_rules(v, prefix)
            else:
                result[k] = prefix_rules(v, prefix)
        return result
    return rule


def prefix_part(part, prefix):
    """Prefix one logic node that sits inside an allOf/anyOf (or is the inner logic of a
    rule). An unconditional part becomes the bare "<prefix>true" token."""
    if part == {}:
        return f"{prefix}true"
    return prefix_rules(part, prefix)


def merge_logic_rule(existing_req, new_raw_rule, raw_rules_list, prefix):
    """Adds a new rule prefixed with 'prefix' to an existing requirement if not already covered.
    
    Args:
        existing_req: The current requirement object (e.g. {"always": {"allOf": ["A"]}}).
        new_raw_rule: The newly calculated rule object for this logic level (unprefixed, full structure).
        raw_rules_list: A list of raw rule objects already accounted for.
        prefix: The prefix string (e.g. 'owg|' or 'hmg|').
    """
    if new_raw_rule in raw_rules_list and new_raw_rule != "never":
        return existing_req
    raw_rules_list.append(new_raw_rule)

    # 1. Handle upgrade from 'never'
    if existing_req == "never" or existing_req == {"always": {"allOf": ["never"]}}:
        return prefix_rules(new_raw_rule, prefix)

    # 2. Extract inner logic components
    def get_logic(r):
        if isinstance(r, dict) and "always" in r:
            return r["always"]
        return r

    logic_existing = get_logic(existing_req)
    logic_new = get_logic(new_raw_rule)

    if logic_existing == logic_new:
        return existing_req

    # 3. Identify new components if the new rule is an anyOf
    new_parts = []
    if isinstance(logic_new, dict) and "anyOf" in logic_new:
        # Determine what already exists in some form
        existing_components = []
        if isinstance(logic_existing, dict) and "anyOf" in logic_existing:
            existing_components = logic_existing["anyOf"]
        else:
            existing_components = [logic_existing]

        for p in logic_new["anyOf"]:
            if p not in existing_components:
                new_parts.append(prefix_part(p, prefix))
    else:
        # New rule is just a single string or allOf, and it differs
        new_parts = [prefix_part(logic_new, prefix)]

    if not new_parts:
        return existing_req

    # 4. Combine into an anyOf structure
    if isinstance(logic_existing, dict) and "anyOf" in logic_existing:
        # Avoid redundant additions to existing anyOf
        for np in new_parts:
            if np not in logic_existing["anyOf"]:
                logic_existing["anyOf"].append(np)
        return existing_req
    else:
        return {"always": {"anyOf": [logic_existing] + new_parts}}


# Track raw rules separately for Open and Inverted to avoid redundant additions
raw_rules_tracker = {
    "Open": {"exits": {}, "locations": {}},
    "Inverted": {"exits": {}, "locations": {}}
}

# Initialize tracker from already processed NG/Inverted-NG logic
for region_name, region_data in entrance_graph.items():
    for exit_name, exit_data in region_data["exits"].items():
        for k in ["Open", "Inverted"]:
            rule = exit_data["requirements"].get(k, "never")
            if exit_name not in raw_rules_tracker[k]["exits"]:
                raw_rules_tracker[k]["exits"][exit_name] = []
            if rule != "never":
                raw_rules_tracker[k]["exits"][exit_name].append(rule)
            
    for loc_name, loc_data in region_data["locations"].items():
        for k in ["Open", "Inverted"]:
            rule = loc_data["requirements"].get(k, "never")
            if loc_name not in raw_rules_tracker[k]["locations"]:
                raw_rules_tracker[k]["locations"][loc_name] = []
            if rule != "never":
                raw_rules_tracker[k]["locations"][loc_name].append(rule)


def process_world_variant(target_world, world_state, prefix, req_key="Open"):
    """Update entrance_graph with a new world variant (OWG or HMG)."""
    for region in target_world.regions:
        if region.name not in entrance_graph:
            continue

        exits = entrance_graph[region.name]["exits"]
        entrances = entrance_graph[region.name]["entrances"]
        locations = entrance_graph[region.name]["locations"]

        # 1. Process Exits
        for exit in region.exits:
            if exit.name.startswith("Mirror From ") or exit.name.startswith("Mirror To "):
                exit.text_access_rule = (
                    f"(|has_mirror|) and ({exit.text_access_rule})"
                    if exit.text_access_rule
                    else "|has_mirror|"
                )
            if exit.connected_region:
                if exit.connected_region.name in ignored_tos:
                    continue
                
                raw_rule = check_rule_keys(exit, state=world_state)
                target = exit.connected_region.name

                # Match an existing exit entry with the SAME name AND SAME target.
                # The base processing splits same-named exits that point to different
                # regions into "<name>" and "<name> (Inverted)". Mirror that here so a
                # glitched exit to a different region does not overwrite the base exit's
                # requirement (which was turning a correct "never" into the glitched rule).
                variant_name = exit.name + " (Inverted)"
                if exit.name in exits and exits[exit.name]["to"] == target:
                    match_name = exit.name
                elif variant_name in exits and exits[variant_name]["to"] == target:
                    match_name = variant_name
                else:
                    match_name = None

                if match_name is not None:
                    # Merge into the matching exit for the specified req_key
                    raw_list = raw_rules_tracker[req_key]["exits"].get(match_name, [])
                    exits[match_name]["requirements"][req_key] = merge_logic_rule(
                        exits[match_name]["requirements"].get(req_key, "never"),
                        raw_rule,
                        raw_list,
                        prefix
                    )
                    raw_rules_tracker[req_key]["exits"][match_name] = raw_list
                else:
                    # New exit, or a new target for an existing name -> "(Inverted)" variant
                    new_name = variant_name if exit.name in exits else exit.name
                    exit_dict = {
                        "to": target,
                        "type": exit.connected_region.type.name,
                        "requirements": {
                            "Open": "never",
                            "Inverted": "never",
                        },
                    }
                    exit_dict["requirements"][req_key] = prefix_rules(raw_rule, prefix)

                    if exit.connected_region.type in [
                        RegionType.Cave,
                        RegionType.Dungeon,
                    ] and region.type in [RegionType.LightWorld, RegionType.DarkWorld]:
                        entrances.append(new_name)
                    exits[new_name] = exit_dict
                    raw_rules_tracker[req_key]["exits"][new_name] = [raw_rule]

                all_rules.add(json.dumps(raw_rule))

        # 2. Process Locations
        for location in region.locations:
            raw_rule = check_rule_keys(location, state=world_state)

            if location.name in locations:
                raw_list = raw_rules_tracker[req_key]["locations"].get(location.name, [])
                locations[location.name]["requirements"][req_key] = merge_logic_rule(
                    locations[location.name]["requirements"].get(req_key, "never"),
                    raw_rule,
                    raw_list,
                    prefix
                )
                raw_rules_tracker[req_key]["locations"][location.name] = raw_list
            else:
                # New location found in this logic level
                locations[location.name] = {
                    "requirements": {
                        "Open": "never",
                        "Inverted": "never",
                    }
                }
                locations[location.name]["requirements"][req_key] = prefix_rules(raw_rule, prefix)
                raw_rules_tracker[req_key]["locations"][location.name] = [raw_rule]
                if "Enemy" in location.name:
                    locations[location.name]["type"] = location.note
            
            all_rules.add(json.dumps(raw_rule))


# === Glitched variants (OWG / HMG) ===
# Each variant world is generated on demand and its reachability is merged into
# entrance_graph as additional "owg|" / "hmg|"-prefixed requirements (the tracker
# gates these on its logic mode; see requirementEvaluator.ts).


from BaseClasses import CollectionState, Entrance as _Entrance
import source.logic.AccessRule as _AR

# Waterwalk (boots, not a bunny) isn't a CollectionState method in OWR; it is the same check as a dash clip.
CollectionState.can_water_walk = lambda self, region, player: self.can_dash_clip(region, player)


def add_glitch_extras(gw):
    """Glitches OWR leaves commented out (OverworldGlitchRules.py), added to the OWG/HMG variant worlds only:
    waterwalk to the Waterfall of Wishing and Zora's Ledge, the Dark World Floating Island clip (gloveless
    Hookshot Cave) and entering GT through the Screen Wrap Clip without crystals."""
    player = 1
    def connect(name, parent, target, rule):
        parent_region = gw.get_region(parent, player)
        if any(e.name == name for e in parent_region.exits):
            return
        ent = _Entrance(player, name, parent_region)
        ent.spot_type = 'OWG'
        parent_region.exits.append(ent)
        ent.connect(gw.get_region(target, player))
        _AR.set_rule(ent, rule)

    zora = gw.get_region('Zora Waterfall Area', player)
    connect('Zora Waterfall Waterwalk', 'Zora Waterfall Area', 'Zora Waterfall Entryway',
            _AR.Primitive('can_water_walk', zora, player))
    _AR.add_rule(gw.get_location("Zora's Ledge", player), _AR.Primitive('can_water_walk', zora, player), combine='or')
    connect('DW Floating Island Clip', 'East Dark Death Mountain (Bottom)', 'Dark Death Mountain Floating Island',
            _AR.Primitive('can_boots_clip_dw', player))
    if gw.mode[player] != 'inverted' and not gw.is_atgt_swapped(player):
        # Only the Screen Wrap Clip reaches GT Stairs without the crystals
        _AR.add_rule(gw.get_entrance('Ganons Tower', player), _AR.Primitive('can_boots_clip_dw', player), combine='or')


def build_variant_world(logic, mode, seeds=range(0, 30)):
    """Generate a glitched variant world, retrying seeds past intermittent
    key-lock generation failures. The extracted access rules are seed-independent
    here (vanilla shuffle), so any seed that generates cleanly yields the same rules."""
    saved = (args.logic[1], args.mode[1])
    args.logic[1] = logic
    args.mode[1] = mode
    try:
        last_err = None
        for seed in seeds:
            try:
                gw = main(args=args, fish=fish, seed=seed)
                add_glitch_extras(gw)
                populate_text_rules(gw)
                return gw
            except RuntimeError as err:
                last_err = err  # usually a transient "Keylock detected" during fill
        raise RuntimeError(f"Could not generate {logic}/{mode} world after seeds "
                           f"{seeds.start}..{seeds.stop - 1}: {last_err}")
    finally:
        args.logic[1], args.mode[1] = saved


def add_variant(logic, mode, prefix, req_key):
    gw = build_variant_world(logic, mode)
    process_world_variant(gw, gw.get_all_state(keys=False), prefix, req_key=req_key)


# Comment out any line below to disable that variant; non-glitched (noglitches /
# inverted) output is unaffected. Generating all four tries several seeds and can
# take a few minutes.
add_variant("owglitches",     "open",     "owg|", "Open")
add_variant("hybridglitches", "open",     "hmg|", "Open")
add_variant("owglitches",     "inverted", "owg|", "Inverted")
add_variant("hybridglitches", "inverted", "hmg|", "Inverted")

ERROR:root:Keylock - can't open locations: 
ERROR:root:code: 011111000000
ERROR:root:Swamp Trench 2 Departure Pot #7
ERROR:root:Swamp I Pot #1
ERROR:root:Swamp Behind Waterfall Pot #8
ERROR:root:Swamp Flooded Room Enemy #5
ERROR:root:Swamp Big Key Ledge Pot #3
ERROR:root:Swamp I Pot #2
ERROR:root:Swamp Attic Enemy #7
ERROR:root:Swamp Attic Pot #4
ERROR:root:Swamp Refill Pot #5
ERROR:root:Swamp Palace - Boss
ERROR:root:Swamp Palace - Waterfall Room
ERROR:root:Swamp Behind Waterfall Enemy #3
ERROR:root:Swamp Attic Pot #1
ERROR:root:Swamp Push Statue Enemy #9
ERROR:root:Swamp I Pot #6
ERROR:root:Swamp Shooters Enemy #4
ERROR:root:Swamp Palace - Flooded Room - Left
ERROR:root:Swamp Right Elbow Enemy #2
ERROR:root:Swamp Waterfall Room Enemy #8
ERROR:root:Swamp I Pot #3
ERROR:root:Swamp Behind Waterfall Pot #10
ERROR:root:Swamp West Shallows Enemy #6
ERROR:root:Swamp Push Statue Pot #5
ERROR:root:Swamp Attic Pot #2
ERROR:root:Swamp Crystal Switch Outer Enemy #5
ERROR:root:Swamp Behind Waterf

In [14]:
from DoorShuffle import default_small_key_doors, key_doors

all_small_key_doors = []
for dp in key_doors:
    all_small_key_doors.append(dp[0])
    all_small_key_doors.append(dp[1])

# 'Eastern Palace': [
#     ('Eastern Dark Square Key Door WN', 'Eastern Cannonball Ledge Key Door EN'),
#     'Eastern Darkness Up Stairs',
# ],
for dungeon, doors in default_small_key_doors.items():
    for door in doors:
        if type(door) == str:
            all_small_key_doors.append(door)
        else:
            for d in door:
                all_small_key_doors.append(d)


all_small_key_doors.remove("Eastern Courtyard N")
all_small_key_doors.remove("Eastern Big Key NE")
all_small_key_doors.remove("Hera Startile Corner NW")
all_small_key_doors.remove("Desert Wall Slide NW")



In [15]:
entrance_graph = json.loads(json.dumps(entrance_graph))  # Deep copy to remove references

In [16]:
never = json.dumps({
    "always": {
        "allOf": [
            "never"
        ]
    }
})
exempt = [
    "Ice Lake Iceberg Bomb Jump",
    "Ice Lake Northeast Pier Hop",
    "Old Man Cave Exit (West)",
    "Paradox Cave Push Block Reverse",
    "Paradox Cave Bomb Jump",
]

big_key_doors = [
    "Hyrule Dungeon Cellblock Door",
    "Eastern Courtyard N",
    "Eastern Big Key NE",
    "Desert Wall Slide NW",
    "Hera Startile Corner NW",
    "PoD Dark Alley NE",
    "Thieves BK Corner NE",
    "Thieves Blind's Cell Door",
    "Ice Crystal Right NE",
    "Mire BK Door Room N",
    "Mire Square Rail NW",
    "Mire Antechamber NW",
    "TR Dodgers NE",
    "TR Final Abyss NW",
    "GT Dash Hall NE",
    "GT Brightly Lit Hall NW",
]

def has_allOf_not_never(req):
    """Check if requirement has always.allOf that is not ['never']"""
    return (isinstance(req, dict) and 
            "always" in req and 
            "allOf" in req["always"] and 
            req["always"]["allOf"] != ['never'])

def has_anyOf(req):
    """Check if requirement has always.anyOf"""
    return (isinstance(req, dict) and 
            "always" in req and 
            "anyOf" in req["always"])

def add_key_requirement(req, key_type):
    """Add a key requirement to existing requirements, handling allOf and anyOf cases"""
    # OWR's rule already names the key (e.g. HMG: "Hera big key, or the Mire big key after the Mire clip");
    # adding it again would cancel the alternative
    if json.dumps(req).count(f'"{key_type}"') > 0:
        return
    if has_allOf_not_never(req):
        req["always"]["allOf"].append(key_type)
    elif has_anyOf(req):
        # Wrap the anyOf in an allOf with the key requirement
        existing_anyOf = req["always"]["anyOf"]
        req["always"] = {
            "allOf": [
                {"anyOf": existing_anyOf},
                key_type
            ]
        }
    else:
        # No existing requirements or requirements are 'never', just set the key
        req.clear()
        req["always"] = {"allOf": [key_type]}

for e, ed in entrance_graph.items():
    for ex, exd in ed['exits'].items():
        # Check if all requirements are "never"
        if exd['requirements']['Open'] == json.loads(never) and exd['requirements']['Inverted'] == json.loads(never):
            print(f"{ex} to {exd['to']} is always never")

for e, ed in entrance_graph.items():
    for ex, exd in ed['exits'].items():
        if ex in big_key_doors:
            add_key_requirement(exd['requirements']['Open'], 'bigkey')
            add_key_requirement(exd['requirements']['Inverted'], 'bigkey')
        elif ex in all_small_key_doors:
            add_key_requirement(exd['requirements']['Open'], 'smallkey')
            add_key_requirement(exd['requirements']['Inverted'], 'smallkey')
        if ex in one_way_doors:
            exd['requirements']['Open'] = json.loads(never)
            exd['requirements']['Inverted'] = json.loads(never)

Ice Lake Iceberg Bomb Jump to Ice Lake Iceberg is always never
Ice Lake Northeast Pier Hop to Ice Lake Northeast Bank is always never
Old Man Cave Exit (West) to Mountain Pass Entry is always never
Paradox Cave Push Block Reverse to Paradox Cave Chest Area is always never
Paradox Cave Bomb Jump to Paradox Cave is always never
PoD Arena Right to Ranged Crystal to PoD Arena Right - Ranged Crystal is always never
PoD Arena Ledge to Ranged Crystal to PoD Arena Ledge - Ranged Crystal is always never
Mire Fishbone Blue Barrier Bypass to Mire South Fish is always never
GT Double Switch Entry to Ranged Switches to GT Double Switch Entry - Ranged Switches is always never
GT Double Switch Left to Exit Bypass to GT Double Switch Exit is always never
GT Double Switch Pot Corners to Ranged Switches to GT Double Switch Pot Corners - Ranged Switches is always never


In [17]:
def add_allOf_requirement(req_dict, requirement):
    if type(req_dict) == str:
        if req_dict == "never":
            req_dict = {}
        else:
            req_dict = {"always": {"allOf": [req_dict]}}
    if "always" not in req_dict:
        req_dict["always"] = {}
    if "allOf" not in req_dict["always"]:
        if "anyOf" in req_dict["always"]:
            # Convert anyOf to allOf with single anyOf inside
            req_dict["always"]["allOf"] = [{"anyOf": req_dict["always"]["anyOf"]}, {"allOf": [requirement]}]
            del req_dict["always"]["anyOf"]
            return
        else:
            req_dict["always"]["allOf"] = []
    if req_dict["always"]["allOf"] == ["never"]:
        req_dict["always"]["allOf"] = []
    if requirement not in req_dict["always"]["allOf"]:
        req_dict["always"]["allOf"].append(requirement)

In [18]:
def add_anyOf_requirement(req_dict, requirement):
    # Use this function to add a new requirements that can be satisfied in place of the existing requirement (i.e. "or")
    if type(req_dict) == str:
        if req_dict == "never":
            req_dict = {}
        else:
            req_dict = {"always": {"anyOf": [req_dict]}}
    if "always" not in req_dict:
        req_dict["always"] = {}
    if "anyOf" not in req_dict["always"]:
        if "allOf" in req_dict["always"]:
            # Convert allOf to anyOf with the existing allOf as one alternative
            req_dict["always"]["anyOf"] = [{"allOf": req_dict["always"]["allOf"]}, requirement]
            del req_dict["always"]["allOf"]
            return
        else:
            req_dict["always"]["anyOf"] = []
    if req_dict["always"]["anyOf"] == ["never"]:
        req_dict["always"]["anyOf"] = []
    if requirement not in req_dict["always"]["anyOf"]:
        req_dict["always"]["anyOf"].append(requirement)


In [19]:
for region, region_data in entrance_graph.items():
    for e in region_data['exits'].keys():
        if e in hasCrystalSwitch:
            entrance_graph[region]['locations']['Crystal_Switch'] = {
                "requirements": {
                    "Open": {
                        "always": {
                            "allOf": [
                                "canHitSwitch"
                            ]
                        }
                    },
                    "Inverted": {
                        "always": {
                            "allOf": [
                                "canHitSwitch"
                            ]
                        }
                    }
                }
            }
        if e in blueBarrier:
            print(f"Adding blueSwitchDown to {e} in {region}")
            open_reqs = entrance_graph[region]['exits'][e]['requirements']['Open']
            inv_reqs = entrance_graph[region]['exits'][e]['requirements']['Inverted']
            add_allOf_requirement(open_reqs, "blueSwitchDown")
            add_allOf_requirement(inv_reqs, "blueSwitchDown")
            print(entrance_graph['Sewers Behind Tapestry']['exits']['Sewers Behind Tapestry S']['requirements']['Open']['always']['allOf'])
        if e in orangeBarrier:
            print(f"Adding orangeSwitchDown to {e} in {region}")
            open_reqs = entrance_graph[region]['exits'][e]['requirements']['Open']
            inv_reqs = entrance_graph[region]['exits'][e]['requirements']['Inverted']
            add_allOf_requirement(open_reqs, "orangeSwitchDown")
            add_allOf_requirement(inv_reqs, "orangeSwitchDown")
            print(entrance_graph['Sewers Behind Tapestry']['exits']['Sewers Behind Tapestry S']['requirements']['Open']['always']['allOf'])


Adding blueSwitchDown to Hera Lobby to Front Barrier - Blue in Hera Lobby
['never']
Adding blueSwitchDown to Hera Front to Lobby Barrier - Blue in Hera Front
['never']
Adding blueSwitchDown to Hera Front to Down Stairs Barrier - Blue in Hera Front
['never']
Adding orangeSwitchDown to Hera Front to Up Stairs Barrier - Orange in Hera Front
['never']
Adding orangeSwitchDown to Hera Front to Back Barrier - Orange in Hera Front
['never']
Adding blueSwitchDown to Hera Front to Back Bypass in Hera Front
['never']
Adding blueSwitchDown to Hera Down Stairs to Front Barrier - Blue in Hera Down Stairs Landing
['never']
Adding orangeSwitchDown to Hera Up Stairs to Front Barrier - Orange in Hera Up Stairs Landing
['never']
Adding orangeSwitchDown to Hera Back to Front Barrier - Orange in Hera Back
['never']
Adding blueSwitchDown to Hera Tile Room EN in Hera Tile Room
['never']
Adding blueSwitchDown to Hera Tridorm WN in Hera Tridorm
['never']
Adding orangeSwitchDown to Hera Tridorm SE in Hera Trido

In [20]:
# Manual fixes go here:

# PoD Fix (logic bug)
entrance_graph["PoD Arena Bridge"]["exits"]["PoD Arena Bridge Drop Down"]["to"] = "PoD Arena Landing"

# Spike cave - complex
# set_rule(world.get_location('Spike Cave', player), lambda state:
#          state.has('Hammer', player) and state.can_lift_rocks(player) and
#          ((state.has('Cape', player) and state.can_extend_magic(player, 16, True)) or
#          (state.has('Cane of Byrna', player) and
#           (state.can_extend_magic(player, 12, True) or (state.world.can_take_damage and (state.has_Boots(player) or state.has_hearts(player, 4)))
#            )))
#          )
spike_requirements = {
    "always": {
        "allOf": [
            "hammer",
            "glove",
        ],
        "anyOf": [
            {
                "allOf": [
                    "cape",
                    "canExtendMagic|16",
                ]
            },
            {
                "allOf": [
                    "byrna",
                    {
                        "anyOf": [
                            "canExtendMagic|12",
                            {
                                "allOf": [
                                    "canTakeDamage",
                                    {
                                        "anyOf": [
                                            "boots",
                                            "hearts|4",
                                        ]
                                    }
                                ]
                            }
                        ]
                    }
                ]
            }
        ],
    }
}

for location in entrance_graph["Spike Cave"]["locations"]:
    entrance_graph["Spike Cave"]["locations"][location]["requirements"]["Open"] = spike_requirements
    entrance_graph["Spike Cave"]["locations"][location]["requirements"]["Inverted"] = spike_requirements


entrance_graph["Mire Area"]["exits"]["Misery Mire"]["requirements"] = {
    "Open": {"always": {"allOf": ["canOpenMM"]}},
    "Inverted": {"always": {"allOf": ["canOpenMM"]}},
}


In [21]:
entrance_graph["Swamp Trench 1 Key Ledge"]

{'exits': {'Swamp Trench 1 Key Ledge Dry': {'to': 'Swamp Trench 1 Nexus',
   'type': 'Dungeon',
   'requirements': {'Open': {}, 'Inverted': {}}},
  'Swamp Trench 1 Key Approach': {'to': 'Swamp Trench 1 Approach',
   'type': 'Dungeon',
   'requirements': {'Open': {'always': {'allOf': ['flippers']}},
    'Inverted': {'always': {'allOf': ['flippers']}}}},
  'Swamp Trench 1 Key Ledge Depart': {'to': 'Swamp Trench 1 Departure',
   'type': 'Dungeon',
   'requirements': {'Open': {'always': {'allOf': ['flippers']}},
    'Inverted': {'always': {'allOf': ['flippers']}}}},
  'Swamp Trench 1 Key Ledge NW': {'to': 'Swamp Hammer Switch',
   'type': 'Dungeon',
   'requirements': {'Open': {'always': {'allOf': [{'anyOf': [{},
         {'allOf': ['hmg|flippers',
           'hmg|otherSmallKeys|mm|6',
           {'anyOf': ['hmg|canDashClip',
             {'allOf': ['hmg|canBombClip', 'hmg|canLightFires']}]},
           'hmg|canReach|Mire Torches Top']}]},
       'smallkey']}},
    'Inverted': {'always': {

In [22]:
# Swamp draining
# The hammer switch drains Trench 1; in HMG so does arriving through the Mire clip (OWR "Trench 1 Switch")
HMG_MIRE_CLIP = {"allOf": ["hmg|canReach|Mire Torches Top", {"anyOf": ["hmg|canDashClip", {"allOf": ["hmg|canBombClip", "hmg|canLightFires"]}]}]}
TRENCH_1_DRAIN = {"anyOf": [{"allOf": ["canReach|Swamp Hammer Switch", "hammer"]}, HMG_MIRE_CLIP]}
# Swamp Hammer Switch
entrance_graph["Swamp Trench 1 Key Ledge"]["exits"]["Swamp Trench 1 Key Ledge Depart"][
    "requirements"
]["Open"]["always"]["allOf"] += [json.loads(json.dumps(TRENCH_1_DRAIN))]

entrance_graph["Swamp Trench 1 Key Ledge"]["exits"]["Swamp Trench 1 Key Ledge Depart"][
    "requirements"
]["Inverted"]["always"]["allOf"] += [json.loads(json.dumps(TRENCH_1_DRAIN))]

entrance_graph["Swamp Trench 1 Approach"]["exits"][
    "Swamp Trench 1 Approach Swim Depart"
]["requirements"]["Open"]["always"]["allOf"] += [json.loads(json.dumps(TRENCH_1_DRAIN))]

entrance_graph["Swamp Trench 1 Approach"]["exits"][
    "Swamp Trench 1 Approach Swim Depart"
]["requirements"]["Inverted"]["always"]["allOf"] += [json.loads(json.dumps(TRENCH_1_DRAIN))]



In [23]:
# Swamp draining

# Swamp Trench 2
# HMG: canHMGMirrorlessSwamp is OWR's moat rule after the Mire clip (Old Man S&Q, Hera/GT leading to the
# Mire/Hera portal); it depends on entrance links, so the tracker evaluates it
_moat = {"always": {"anyOf": [{"allOf": ["flippers", "canDrainDam"]}, "hmg|canHMGMirrorlessSwamp"]}}
entrance_graph["Swamp Lobby"]["exits"]["Swamp Lobby Moat"]["requirements"] = {"Open": _moat, "Inverted": json.loads(json.dumps(_moat))}


add_allOf_requirement(entrance_graph["Swamp Trench 2 Pots"]["exits"]["Swamp Trench 2 Pots Wet"][
    "requirements"
]["Open"], "canReach|Swamp Crystal Switch Inner")

add_allOf_requirement(entrance_graph["Swamp Trench 2 Pots"]["exits"]["Swamp Trench 2 Pots Wet"][
    "requirements"
]["Inverted"], "canReach|Swamp Crystal Switch Inner")



add_allOf_requirement(
    entrance_graph["Ice Jelly Key"]["exits"]["Ice Jelly Key Down Stairs"][
        "requirements"
    ]["Open"],
    "smallkey",
)

add_allOf_requirement(
    entrance_graph["Ice Jelly Key"]["exits"]["Ice Jelly Key Down Stairs"][
        "requirements"
    ]["Inverted"],
    "smallkey",
)


# Inverted region fixes
entrance_graph["Big Bomb Shop"]["exits"]["Big Bomb Shop Exit"] = {
    "to": "Big Bomb Shop Area",
    "type": "DarkWorld",
    "requirements": {"Open": {}, "Inverted": {}},
}

entrance_graph["Dark Sanctuary Hint"]["exits"]["Dark Sanctuary Hint Exit"] = {
    "to": "Dark Chapel Area",
    "type": "DarkWorld",
    "requirements": {"Open": {}, "Inverted": {}},
}

# Ice Palace logic
for i in ["Open", "Inverted"]:
    entrance_graph["Ice Crystal Left"]["exits"]["Ice Crystal Left Blue Barrier"][
        "requirements"
    ][i] = {"always": {"allOf": ["canHitSwitch", "canReach|Ice Refill"]}}

# One way trap doors
entrance_graph["GT Bob's Torch"]["exits"]["GT Torch SW"]["requirements"] = {
    "Open": {"always": {"allOf": ["never"]}},
    "Inverted": {"always": {"allOf": ["never"]}},
}

entrance_graph["Mire Wizzrobe Bypass"]["exits"]["Mire Wizzrobe Bypass WN"][
    "requirements"
] = {
    "Open": {"always": {"allOf": ["never"]}},
    "Inverted": {"always": {"allOf": ["never"]}},
}

entrance_graph["Ice Hookshot Ledge"]["exits"]["Ice Hookshot Ledge WN"][
    "requirements"
] = {
    "Open": {"always": {"allOf": ["never"]}},
    "Inverted": {"always": {"allOf": ["never"]}},
}

entrance_graph["GT Bob's Torch"]["exits"]["GT Torch SW"]["requirements"] = {
    "Open": {"always": {"allOf": ["never"]}},
    "Inverted": {"always": {"allOf": ["never"]}},
}

# Old man exit broken
entrance_graph["Old Man Cave Ledge"]["exits"]["Old Man Cave Exit (West)"]["requirements"]["Open"] = {}
entrance_graph["Old Man Cave Ledge"]["exits"]["Old Man Cave Exit (West)"]["requirements"]["Inverted"] = {}

In [24]:
entrance_graph["Swamp Trench 1 Key Ledge"]["exits"]["Swamp Trench 1 Key Ledge Depart"][
    "requirements"
]["Open"]["always"]

{'allOf': ['flippers',
  {'anyOf': [{'allOf': ['canReach|Swamp Hammer Switch', 'hammer']},
    {'allOf': ['hmg|canReach|Mire Torches Top',
      {'anyOf': ['hmg|canDashClip',
        {'allOf': ['hmg|canBombClip', 'hmg|canLightFires']}]}]}]}]}

In [25]:

# Inverted 1.0

# - Links House start now spawns in Big Bomb Shop - DONE
# Fix menu exit
entrance_graph["Menu"]["exits"]["Links House S&Q (Inverted)"]['requirements']['Inverted_1'] = "never"
entrance_graph["Menu"]["exits"]["Links House S&Q"]['requirements']['Inverted_1'] = {}

# Relink link's house 
entrance_graph["Links House Area"]["exits"]["Links House"]['requirements']['Inverted_1'] = "never"
entrance_graph["Links House"]["exits"]["Links House Exit"]['requirements']['Inverted_1'] = "never"
entrance_graph["Links House"]["exits"]["Links House Exit (Inverted_1)"] = {
    "to": "Big Bomb Shop Area",
    "type": "DarkWorld",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}

    }
}
entrance_graph["Links House Area"]["exits"]["Big Bomb Shop"] = {
    "to": "Big Bomb Shop",
    "type": "Cave",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}

# Relink Big Bomb Shop
entrance_graph["Big Bomb Shop Area"]["exits"]["Big Bomb Shop"]['requirements']['Inverted_1'] = "never"
entrance_graph["Big Bomb Shop"]["exits"]["Big Bomb Shop Exit"]['requirements']['Inverted_1'] = "never"
entrance_graph["Big Bomb Shop"]["exits"]["Big Bomb Shop Exit (Inverted_1)"] = {
    "to": "Links House Area",
    "type": "LightWorld",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}
entrance_graph["Big Bomb Shop Area"]["exits"]["Links House"] = {
    "to": "Links House",
    "type": "Cave",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}


# - Mountain Cave S+Q now spawns in his usual Cave - DONE
entrance_graph["Menu"]["exits"]["Old Man S&Q"]['requirements']['Inverted_1'] = "never"
entrance_graph["Menu"]["exits"]["Old Man S&Q (Inverted_1)"] = {
    "to": "West Dark Death Mountain (Bottom)",
    "type": "DarkWorld",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {
            "always": {
                "allOf": [
                    "canRescueOldMan"
                ]
            }
        }
    }
}

# - Flute Spot 1 is moved to Top of Dark Death Mountain - DONE
entrance_graph["Flute Sky"]["exits"]["Flute Spot 1 (Inverted)"]['requirements']['Inverted_1'] = "never"
entrance_graph["Flute Sky"]["exits"]["Flute Spot 1 (Inverted_1)"] = {
    "to": "West Dark Death Mountain (Bottom)",
    "type": "DarkWorld",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {
            "always": {
                "allOf": [
                    "canFlute"
                ]
            }
        }
    }
}


# - Spiral and Mimic Cave are now bridged together - DONE
# - Just make all of these exits not possible in Inv1
for exit in entrance_graph["Spiral Mimic Ledge Extend"]["exits"]:
    entrance_graph["Spiral Mimic Ledge Extend"]["exits"][exit]['requirements']['Inverted_1'] = "never"

# Re-add the inverted dropdpown
entrance_graph["East Death Mountain (Top East)"]['exits']['Mimic Cave Drop (Inverted_1)'] = {
    "to": "Mimic Cave Ledge",
    "type": "LightWorld",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}

# - Ladder is removed from West Dark Death Mountain - DONE
# Reverse way is just a dropdown and already exists
entrance_graph["West Dark Death Mountain (Bottom)"]['exits']['West Dark Death Mountain Ladder (Inverted_1)'] = {
    "to": "West Dark Death Mountain (Top)",
    "type": "DarkWorld",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}


# - TR Peg Puzzle is restored but instead reveals a ladder
# - Not needed. The requirements are the same anyway (mitts + hammer)

# - Old Man Cave/Bumper Cave returned to vanilla
# Relink Bumper cave bottom
entrance_graph["Bumper Cave Entry"]["exits"]["Bumper Cave (Bottom)"]['requirements']['Inverted_1'] = "never"
entrance_graph["Bumper Cave Entry"]["exits"]["Bumper Cave (Bottom Inverted_1)"] = {
    "to": "Old Man Cave Ledge",
    "type": "Cave",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}
entrance_graph["Old Man Cave Ledge"]["exits"]["Old Man Cave Exit (West)"]["requirements"]["Inverted_1"] = "never"
entrance_graph["Old Man Cave Ledge"]["exits"]["Old Man Cave Exit (West Inverted_1)"] = {
    "to": "Bumper Cave Entry",
    "type": "DarkWorld",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}

# Relink DWWDM Fairy
entrance_graph["West Dark Death Mountain (Bottom)"]["exits"]["Dark Death Mountain Fairy"]['requirements']['Inverted_1'] = "never"
entrance_graph["West Dark Death Mountain (Bottom)"]["exits"]["Dark Death Mountain Fairy (Inverted_1)"] = {
    "to": "Old Man Cave (East)",
    "type": "Cave",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}

entrance_graph["Old Man Cave (East)"]["exits"]["Old Man Cave Exit (East)"]["requirements"]["Inverted_1"] = "never"
entrance_graph["Old Man Cave (East)"]["exits"]["Old Man Cave Exit (East Inverted_1)"] = {
    "to": "West Dark Death Mountain (Bottom)",
    "type": "DarkWorld",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {
            "always": {
                "allOf": [
                    "lantern"
                ]
            }
        }
    }
}

# Relink old man cave
entrance_graph["West Death Mountain (Bottom)"]["exits"]["Old Man Cave (East)"]['requirements']['Inverted_1'] = "never"
entrance_graph["West Death Mountain (Bottom)"]["exits"]["Old Man Cave (East Inverted_1)"] = {
    "to": "Death Mountain Return Cave (left)",
    "type": "Cave",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}

entrance_graph["Death Mountain Return Cave (left)"]["exits"]["Death Mountain Return Cave Exit (West)"]['requirements']['Inverted_1'] = "never"
entrance_graph["Death Mountain Return Cave (left)"]["exits"]["Death Mountain Return Cave Exit (West Inverted_1)"] = {
    "to": "West Death Mountain (Bottom)",
    "type": "LightWorld",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}

# Relink DMA cave
entrance_graph["Mountain Pass Entry"]["exits"]["Old Man Cave (West)"]['requirements']['Inverted_1'] = "never"
entrance_graph["Mountain Pass Entry"]["exits"]["Old Man Cave (West Inverted_1)"] = {
    "to": "Bumper Cave (bottom)",
    "type": "Cave",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}
entrance_graph["Bumper Cave (bottom)"]["exits"]["Bumper Cave Exit (Bottom)"]['requirements']['Inverted_1'] = "never"
entrance_graph["Bumper Cave (bottom)"]["exits"]["Bumper Cave Exit (Inverted_1)"] = {
    "to": "Mountain Pass Entry",
    "type": "LightWorld",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}

# Relink Bumper Cave top
entrance_graph["Bumper Cave Ledge"]["exits"]["Bumper Cave (Top)"]['requirements']['Inverted_1'] = "never"
entrance_graph["Bumper Cave Ledge"]["exits"]["Bumper Cave (Top Inverted_1)"] = {
    "to": "Dark Death Mountain Healer Fairy",
    "type": "Cave",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}

# Relink DMA Cave lower
entrance_graph["Mountain Pass Ledge"]["exits"]["Death Mountain Return Cave (West)"]['requirements']['Inverted_1'] = "never"
entrance_graph["Mountain Pass Ledge"]["exits"]["Death Mountain Return Cave (West Inverted_1)"] = {
    "to": "Bumper Cave (top)",
    "type": "Cave",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}

# Relink DMD cave right
entrance_graph["Bumper Cave (top)"]["exits"]["Bumper Cave Exit (Top)"]['requirements']['Inverted_1'] = "never"
entrance_graph["Bumper Cave (top)"]["exits"]["Bumper Cave Exit (Top Inverted_1)"] = {
    "to": "Mountain Pass Ledge",
    "type": "LightWorld",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}


# - Ice Palace has been re-sealed to vanilla, portal moved to outer edge of moat
entrance_graph["Ice Lake Iceberg"]["exits"]["Ice Palace (Inverted)"] = {
    "to": "Ice Palace Area",
    "type": "DarkWorld",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}

entrance_graph["Ice Palace Area"]["exits"]["Ice Lake Iceberg"] = {
    "to": "Ice Lake Iceberg",
    "type": "DarkWorld",
    "requirements": {
        "Open": "never",
        "Inverted": "never",
        "Inverted_1": {}
    }
}

entrance_graph["Ice Lake Water"]["exits"]["Ice Lake Iceberg"] = {
    "to": "Ice Lake Iceberg",
    "type": "DarkWorld",
    "requirements": {
        "Open": {
            "always": {
                "allOf": [
                    "moonpearl"
                    "flippers"
                ]
            }
        },
        "Inverted": {
            "always": {
                "allOf": [
                    "flippers"
                ]
            }
        },
        "Inverted_1":{
            "always": {
                "allOf": [
                    "flippers"
                ]
            }
        }
    }
}

entrance_graph["Ice Lake Iceberg"]["exits"]["Ice Lake Water"] = {
    "to": "Ice Lake Water",
    "type": "DarkWorld",
    "requirements": {
        "Open": {
            "always": {
                "allOf": [
                    "moonpearl"
                    "flippers"
                ]
            }
        },
        "Inverted": {
            "always": {
                "allOf": [
                    "flippers"
                ]
            }
        },
        "Inverted_1":{
            "always": {
                "allOf": [
                    "flippers"
                ]
            }
        }
    }
}

In [26]:
# Add boss kill requirements to Boss locations
for region_name, region_data in entrance_graph.items():
    for exit_name, exit_data in region_data["exits"].items():
        if exit_data["to"].endswith(" Boss Spoils"):
            add_allOf_requirement(
                exit_data["requirements"]["Open"], "canKillBoss"
            )
            add_allOf_requirement(
                exit_data["requirements"]["Inverted"], "canKillBoss"
            )

In [27]:
# Ice Armos is a kill room
add_allOf_requirement(
    entrance_graph["GT Ice Armos"]["exits"]["GT Ice Armos NE"]['requirements']['Open'], "canKillBoss|armos"
)
add_allOf_requirement(
    entrance_graph["GT Ice Armos"]["exits"]["GT Ice Armos NE"]['requirements']['Inverted'], "canKillBoss|armos"
)

# Lanmo 2 is a kill room
add_allOf_requirement(
    entrance_graph["GT Lanmolas 2"]["exits"]["GT Lanmolas 2 NW"]['requirements']['Open'], "canKillBoss|lanmolas"
)
add_allOf_requirement(
    entrance_graph["GT Lanmolas 2"]["exits"]["GT Lanmolas 2 NW"]['requirements']['Inverted'], "canKillBoss|lanmolas"
)

# Validation chest doesn't show without killing moldorm 2
add_allOf_requirement(
    entrance_graph["GT Validation"]["locations"]["Ganons Tower - Validation Chest"]['requirements']['Open'], "canKillBoss|moldorm"
)
add_allOf_requirement(
    entrance_graph["GT Validation"]["locations"]["Ganons Tower - Validation Chest"]['requirements']['Inverted'], "canKillBoss|moldorm"
)

In [28]:
from Rules import std_kill_rooms, kill_chests

for region_name, region_data in std_kill_rooms.items():
    blocked_exits = region_data[0]
    for exit_name in blocked_exits:
        if exit_name in entrance_graph[region_name]["exits"]:
            add_allOf_requirement(
                entrance_graph[region_name]["exits"][exit_name]['requirements']['Open'], "canClearRoom"
            )
            add_allOf_requirement(
                entrance_graph[region_name]["exits"][exit_name]['requirements']['Inverted'], "canClearRoom"
            )

for region_name, region_data in kill_chests.items():
    for location_name in region_data[0]:
        if location_name in entrance_graph[region_name]["locations"]:
            add_allOf_requirement(
                entrance_graph[region_name]["locations"][location_name]['requirements']['Open'], "canClearRoom"
            )
            add_allOf_requirement(
                entrance_graph[region_name]["locations"][location_name]['requirements']['Inverted'], "canClearRoom"
            )

In [29]:
# Remove any location that ends with "Crab Drop" since these are just for logic purposes and not actual locations
# Remove any location that ends with "Tree Pull" since these are just for logic purposes and not actual locations
for region_name, region_data in entrance_graph.items():
    locations_to_remove = []
    for location_name in region_data["locations"]:
        if location_name.endswith("Crab Drop") or location_name.endswith("Tree Pull"):
            locations_to_remove.append(location_name)
    for loc in locations_to_remove:
        del entrance_graph[region_name]["locations"][loc]

In [30]:
#  Logic breaks
# Icebreaker
entrance_graph["Ice Dead End"]["exits"]["Ice Dead End Icebreaker"] = {
    "to": "Ice Big Key",
    "type": "Dungeon",
    "requirements": {
        "Open": {
            "always": {
                "allOf": [
                    "canIceBreak"
                ]
            }
        },
        "Inverted": {
            "always": {
                "allOf": [
                    "canIceBreak"
                ]
            }
        }
    }
}

In [31]:
# Hovering
entrance_graph["Spectacle Rock Cave (Peak)"]["exits"]["Spec Rock Hover"] = {
    "to": "Spectacle Rock Cave (Top)",
    "type": "Cave",
    "requirements": {
        "Open": {
            "always": {
                "allOf": [
                    "canHover"
                ]
            }
        },
        "Inverted": {
            "always": {
                "allOf": [
                    "canHover"
                ]
            }
        }
    }
}

add_anyOf_requirement(entrance_graph["GT Moldorm"]["exits"]["GT Moldorm Gap"]['requirements']['Open'], "canHover")
add_anyOf_requirement(entrance_graph["GT Moldorm"]["exits"]["GT Moldorm Gap"]['requirements']['Inverted'], "canHover")

add_anyOf_requirement(entrance_graph["Hookshot Cave (Front)"]["exits"]["Hookshot Cave Hook Path"]['requirements']['Open'], "canHover")
add_anyOf_requirement(entrance_graph["Hookshot Cave (Front)"]["exits"]["Hookshot Cave Hook Path"]['requirements']['Inverted'], "canHover")


In [32]:
# Bomb jumps
entrance_graph["Spectacle Rock Cave (Top)"]["exits"]["Spec Rock Bomb Jump"] = {
    "to": "Spectacle Rock Cave (Peak)",
    "type": "Cave",
    "requirements": {
        "Open": {
            "always": {
                "allOf": [
                    "canBombJump"
                ]
            }
        },
        "Inverted": {
            "always": {
                "allOf": [
                    "canBombJump"
                ]
            }
        }
    }
}

entrance_graph["Skull 1 Lobby"]["exits"]["SW Big Chest Bomb Jump"] = {
    "to": "Skull Big Chest",
    "type": "Dungeon",
    "requirements": {
        "Open": {
            "always": {
                "allOf": [
                    "canBombJump"
                ]
            }
        },
        "Inverted": {
            "always": {
                "allOf": [
                    "canBombJump"
                ]
            }
        }
    }
}

In [33]:
# HC Main
add_anyOf_requirement(entrance_graph["Hyrule Castle Area"]["exits"]["Hyrule Castle Main Gate (South)"]['requirements']['Open'], "canReachHCMain")


In [34]:
#  Glitched

In [35]:
# === Clean up output rules (graph-aware prune + mode-superset dedup) ===
# Two independent, provably accessibility-preserving simplifications applied to the
# final entrance_graph. Each has its own toggle (set False / comment out to skip);
# non-glitched output is unaffected either way.
#
# 1) PRUNE_REDUNDANT_GLITCH_PATHS: the glitched variants (owg|/hmg|) bake full
#    dungeon-reentry PATHS into location/exit rules (e.g. PoD via the Kiki Skip clip:
#    Reach(clip) + interior door/crystal/enemy traversal). Those are redundant: the
#    clip EDGES are already in the graph, so the traverser reaches the spot's region
#    on its own. We drop an anyOf branch iff it contains a glitched canReach AND the
#    spot's region is already reachable via graph EDGES under that logic mode.
#
# 2) DEDUPE_MODE_REDUNDANT_ALTERNATIVES: logic modes are supersets
#    (noglitches < owg < hmg), and the tracker evaluates a glitched mode as including
#    all lower-mode rules. So within an anyOf, a branch is redundant when a sibling
#    has the SAME mode-stripped form at a lower-or-equal mode (e.g. drop
#    "hmg|canHitRangedSwitch" when "canHitRangedSwitch" is already a sibling).
PRUNE_REDUNDANT_GLITCH_PATHS = True
DEDUPE_MODE_REDUNDANT_ALTERNATIVES = True

_PMODES = {"base": 0, "owg": 1, "hmg": 2}
_PROOTS = ["Menu", "Flute Sky", "Chris Houlihan Room"]

def _p_leaf_mode(t):
    return "owg" if t.startswith("owg|") else ("hmg" if t.startswith("hmg|") else "base")

def _p_strip_mode(t):
    for p in ("owg|", "hmg|"):
        if t.startswith(p):
            return t[len(p):]
    return t

def _p_node_mode(node):
    best = [0]
    def rec(o):
        if isinstance(o, str):
            best[0] = max(best[0], _PMODES[_p_leaf_mode(o)])
        elif isinstance(o, dict):
            for k, v in o.items():
                if k in ("always", "allOf", "anyOf"):
                    rec(v)
        elif isinstance(o, list):
            for v in o:
                rec(v)
    rec(node)
    return best[0]

def _p_eval(node, mode_level, reachable):
    """Edge/requirement truth with items assumed available; glitch tokens gated by
    mode_level; canReach|X -> X in reachable; 'never'->False; {}->True."""
    if node == {} or node is None:
        return True
    if node == "never":
        return False
    if isinstance(node, str):
        if _PMODES[_p_leaf_mode(node)] > mode_level:
            return False
        base = _p_strip_mode(node)
        if base.startswith("canReach|") or base.startswith("canBreach|"):
            return base.split("|", 1)[1] in reachable
        return True
    if isinstance(node, dict):
        if "always" in node:
            return _p_eval(node["always"], mode_level, reachable)
        ok = True
        if "allOf" in node:
            ok = all(_p_eval(x, mode_level, reachable) for x in node["allOf"])
            if not ok:
                return False
        if "anyOf" in node:
            ok = ok and any(_p_eval(x, mode_level, reachable) for x in node["anyOf"])
        return ok
    return True

def _p_reachable(graph, key, mode_level):
    reach = set(r for r in _PROOTS if r in graph)
    changed = True
    while changed:
        changed = False
        for r in list(reach):
            for _en, xv in graph.get(r, {}).get("exits", {}).items():
                to = xv.get("to")
                if not to or to in reach:
                    continue
                if _p_eval(xv.get("requirements", {}).get(key, "never"), mode_level, reach):
                    reach.add(to)
                    changed = True
    return reach

def _p_branch_glitch_canreach_mode(node, region=None):
    """Highest glitch mode of a canReach/canBreach leaf in the branch. A reach of the
    spot's own region is trivially satisfied, not a reentry path, so it is ignored."""
    best = [None]
    def rec(o):
        if isinstance(o, str):
            lm = _p_leaf_mode(o)
            base = _p_strip_mode(o)
            if (lm != "base" and base.startswith(("canReach|", "canBreach|"))
                    and base.split("|", 1)[1] != region):
                if best[0] is None or _PMODES[lm] > _PMODES[best[0]]:
                    best[0] = lm
        elif isinstance(o, dict):
            for k, v in o.items():
                if k in ("always", "allOf", "anyOf"):
                    rec(v)
        elif isinstance(o, list):
            for v in o:
                rec(v)
    rec(node)
    return best[0]

def _p_atoms(node, drop_glitch_reach_except=None):
    """All leaf tokens, mode-stripped. With drop_glitch_reach_except set, glitched canReach leaves (other
    than of that region) are left out: they are the re-entry path, not a requirement of the spot."""
    out = set()
    def rec(o):
        if isinstance(o, str):
            base = _p_strip_mode(o)
            if (drop_glitch_reach_except is not None and _p_leaf_mode(o) != "base"
                    and base.startswith(("canReach|", "canBreach|")) and base.split("|", 1)[1] != drop_glitch_reach_except):
                return
            out.add(base)
        elif isinstance(o, dict):
            for k, v in o.items():
                if k in ("always", "allOf", "anyOf"):
                    rec(v)
        elif isinstance(o, list):
            for v in o:
                rec(v)
    rec(node)
    return out

def _p_subsumed(branch, siblings, region):
    """A glitched re-entry branch is redundant only when a sibling without a glitched reach needs no more
    than the branch does besides its path (e.g. the same interior rule). Otherwise it is a real alternative,
    like a Mire clip instead of the hammer, and must stay."""
    need = _p_atoms(branch, drop_glitch_reach_except=region)
    for s in siblings:
        if s is branch or _p_branch_glitch_canreach_mode(s, region) is not None:
            continue
        if _p_atoms(s) <= need:
            return True
    return False

def _p_prune(node, region, key, reach):
    if isinstance(node, dict):
        out = {}
        for k, v in node.items():
            if k == "anyOf" and isinstance(v, list):
                kept = []
                for b in v:
                    gm = _p_branch_glitch_canreach_mode(b, region)
                    if gm is not None and region in reach[key][gm] and _p_subsumed(b, v, region):
                        continue  # redundant glitched reentry path
                    kept.append(_p_prune(b, region, key, reach))
                # An emptied anyOf would simplify to {} (always true); keep the
                # original alternatives instead.
                out[k] = kept if kept else [_p_prune(b, region, key, reach) for b in v]
            elif k in ("always", "allOf"):
                out[k] = _p_prune(v, region, key, reach)
            else:
                out[k] = v
        return out
    if isinstance(node, list):
        return [_p_prune(v, region, key, reach) for v in node]
    return node

def _p_canon(node):
    """Mode-stripped, order-independent canonical form (hashable)."""
    if isinstance(node, str):
        return _p_strip_mode(node)
    if isinstance(node, dict):
        if "always" in node:
            return _p_canon(node["always"])
        parts = []
        if "allOf" in node:
            parts.append(("allOf", tuple(sorted((_p_canon(x) for x in node["allOf"]), key=repr))))
        if "anyOf" in node:
            parts.append(("anyOf", tuple(sorted((_p_canon(x) for x in node["anyOf"]), key=repr))))
        return parts[0] if len(parts) == 1 else tuple(parts)
    return ("empty",)

def _p_pure_atoms(branch):
    """If branch is a leaf or an allOf of leaves, return (mode-stripped atom set,
    mode); else None (not a pure conjunction -> skip absorption for it)."""
    if isinstance(branch, str):
        return frozenset([_p_strip_mode(branch)]), _PMODES[_p_leaf_mode(branch)]
    if (isinstance(branch, dict) and set(branch.keys()) == {"allOf"}
            and all(isinstance(x, str) for x in branch["allOf"])):
        atoms = frozenset(_p_strip_mode(x) for x in branch["allOf"])
        mode = max((_PMODES[_p_leaf_mode(x)] for x in branch["allOf"]), default=0)
        return atoms, mode
    return None

def _p_absorb(node):
    """Remove dominated anyOf branches. Branch A dominates branch B (drop B) when A's
    mode-stripped atoms are a subset of B's AND mode(A) <= mode(B) -- i.e. B implies A,
    so (A or B) == A. This covers mode-superset duplicates (drop hmg|X when X is a
    sibling, since hmg includes owg includes noglitches) and logical absorption (drop
    owg|hammer & owg|canBunnyPocket when owg|hammer alone is a sibling). Branches that
    are not pure conjunctions fall back to exact-canonical-form dedup."""
    if isinstance(node, dict):
        out = {}
        for k, v in node.items():
            if k == "anyOf" and isinstance(v, list):
                branches = [_p_absorb(b) for b in v]
                info = [_p_pure_atoms(b) for b in branches]
                keep = [True] * len(branches)
                for i in range(len(branches)):
                    if not keep[i]:
                        continue
                    for j in range(len(branches)):
                        if i == j or not keep[j]:
                            continue
                        ai, aj = info[i], info[j]
                        if ai is None or aj is None:
                            if (_p_canon(branches[i]) == _p_canon(branches[j])
                                    and _p_node_mode(branches[i]) <= _p_node_mode(branches[j])
                                    and (i < j or _p_node_mode(branches[i]) < _p_node_mode(branches[j]))):
                                keep[j] = False
                            continue
                        si, mi = ai
                        sj, mj = aj
                        if si <= sj and mi <= mj and (si < sj or mi < mj or i < j):
                            keep[j] = False
                out[k] = [b for b, kp in zip(branches, keep) if kp]
            elif k in ("always", "allOf"):
                out[k] = _p_absorb(v)
            else:
                out[k] = v
        return out
    if isinstance(node, list):
        return [_p_absorb(v) for v in node]
    return node

def _p_simplify(node):
    if isinstance(node, dict):
        if "always" in node:
            inner = _p_simplify(node["always"])
            if inner == {} or inner == "":
                return {}
            if isinstance(inner, str):
                inner = {"allOf": [inner]}
            return {"always": inner}
        out = {}
        if "allOf" in node:
            items = [_p_simplify(x) for x in node["allOf"]]
            flat = []
            for i in items:
                if isinstance(i, dict) and set(i.keys()) == {"allOf"}:
                    flat.extend(i["allOf"])
                elif i == {} or i == "":
                    continue
                else:
                    flat.append(i)
            uniq = []
            for i in flat:
                if i not in uniq:
                    uniq.append(i)
            if "never" in uniq:
                return "never"
            if uniq:
                out["allOf"] = uniq
        if "anyOf" in node:
            items = [_p_simplify(x) for x in node["anyOf"]]
            flat = []
            for i in items:
                if isinstance(i, dict) and set(i.keys()) == {"anyOf"}:
                    flat.extend(i["anyOf"])
                else:
                    flat.append(i)
            flat = [i for i in flat if i != "never"]
            uniq = []
            for i in flat:
                if i not in uniq:
                    uniq.append(i)
            if any(i == {} for i in uniq):
                if "allOf" in out:
                    return {"allOf": out["allOf"]} if len(out["allOf"]) != 1 else out["allOf"][0]
                return {}
            if uniq:
                out["anyOf"] = uniq
        if set(out.keys()) == {"allOf"} and len(out["allOf"]) == 1:
            return out["allOf"][0]
        if set(out.keys()) == {"anyOf"} and len(out["anyOf"]) == 1:
            return out["anyOf"][0]
        return out
    return node

def cleanup_output_rules(graph):
    keys = set()
    for rd in graph.values():
        for coll in ("exits", "locations"):
            for sv in rd.get(coll, {}).values():
                keys.update(sv.get("requirements", {}).keys())
    reach = {k: {m: _p_reachable(graph, k, ml) for m, ml in _PMODES.items()} for k in keys}
    before = {k: _p_reachable(graph, k, _PMODES["hmg"]) for k in keys}

    before_chars = after_chars = 0
    for rn, rd in graph.items():
        for coll in ("exits", "locations"):
            for sv in rd.get(coll, {}).values():
                reqs = sv.get("requirements", {})
                for key in list(reqs.keys()):
                    node = reqs[key]
                    before_chars += len(json.dumps(node))
                    if PRUNE_REDUNDANT_GLITCH_PATHS:
                        node = _p_prune(node, rn, key, reach)
                    # Simplify first so nested allOf/anyOf are flattened and
                    # mode-redundant siblings become directly comparable for dedup.
                    node = _p_simplify(node)
                    if DEDUPE_MODE_REDUNDANT_ALTERNATIVES:
                        # Iterate absorb+simplify to a fixpoint: each simplify pass can
                        # flatten a nested branch into a pure conjunction that only then
                        # becomes comparable for absorption.
                        for _ in range(6):
                            new_node = _p_simplify(_p_absorb(node))
                            if new_node == node:
                                break
                            node = new_node
                    reqs[key] = node
                    after_chars += len(json.dumps(reqs[key]))
    # Guard: edge-reachability must be unchanged.
    after = {k: _p_reachable(graph, k, _PMODES["hmg"]) for k in keys}
    for k in keys:
        lost = before[k] - after[k]
        assert not lost, f"Cleanup changed reachability for {k}: lost {sorted(lost)[:5]}"
    pct = 100 * (before_chars - after_chars) // max(before_chars, 1)
    print(f"Cleaned output rules: requirement chars {before_chars} -> {after_chars} ({pct}% smaller); reachability preserved.")

if PRUNE_REDUNDANT_GLITCH_PATHS or DEDUPE_MODE_REDUNDANT_ALTERNATIVES:
    cleanup_output_rules(entrance_graph)


Cleaned output rules: requirement chars 674844 -> 178061 (73% smaller); reachability preserved.


In [36]:
with open("all_rules_parsed.txt", "w") as f:
    for rule in sorted(all_rules):
        f.write(rule + "\n")

In [37]:
with open("logic_regions.ts", "w") as f:
    f.write("// This file is auto-generated by generate_overworld_entrances.py\n")
    f.write('import type { RegionLogic } from "@/data/logic/logicTypes";\n\n')
    f.write('export const logic_regions: Record<string, RegionLogic> = ')
    import json
    f.write(json.dumps(entrance_graph, indent=4))
    f.write(";\n")

In [38]:
with open("../alttptracker_react/src/data/logic/logic_regions.ts", "w") as f:
    # // This file is auto-generated by generate_overworld_entrances.py
    # import type { OverworldRegionLogic } from "@/data/logic/logicTypes";

    # export const logic_regions: Record<string, OverworldRegionLogic> = {
    f.write("// This file is auto-generated by generate_overworld_entrances.py\n")
    f.write('import type { RegionLogic } from "@/data/logic/logicTypes";\n\n')
    f.write('export const logic_regions: Record<string, RegionLogic> = ')
    import json
    f.write(json.dumps(entrance_graph, indent=4))
    f.write(";\n")

In [39]:
import json
import os
import sys

# Add the local directory to sys.path so we can import local modules
sys.path.append(os.getcwd())

from OWEdges import OWEdgeGroups, parallel_links, OWTileRegions

def extract_ow_data():
    # 1. Similar edge groups
    similar_edge_groups = []
    group_id_counter = 0
    
    # We iterate over OWEdgeGroups and look for multi-edge lists
    for key, (forward_lists, back_lists) in OWEdgeGroups.items():
        # forward_lists is a list of lists, e.g. [['Edge1', 'Edge2'], ['Edge3']]
        for edge_list in forward_lists + back_lists:
            if len(edge_list) > 1:
                group_id_counter += 1
                similar_edge_groups.append(edge_list)

    # 2. Whirlpool registry
    # Hand-picked from OWEdges.py search but can be automated if we find the list
    whirlpools = [
        "Zora Whirlpool",
        "Kakariko Pond Whirlpool",
        "River Bend Whirlpool",
        "C Whirlpool",
        "Lake Hylia Whirlpool",
        "Octoballoon Whirlpool",
        "Qirn Jump Whirlpool",
        "Bomber Corner Whirlpool"
    ]

    # 3. Flute spot registry
    # Extracted from logic_regions.ts manually or by code. 
    # Based on the read_file output:
    flute_spots = {
        1: {"region": "West Death Mountain (Bottom)", "owid": 0x03},
        2: {"region": "Potion Shop Area", "owid": 0x16},
        3: {"region": "Kakariko Village", "owid": 0x18},
        4: {"region": "Links House Area", "owid": 0x2b},
        5: {"region": "Eastern Nook Area", "owid": 0x1e}, # Actually Eastern Palace area owid
        6: {"region": "Desert Teleporter Ledge", "owid": 0x30},
        7: {"region": "Dam Area", "owid": 0x3b},
        8: {"region": "Octoballoon Area", "owid": 0x3f}
    }

    # 4. Default tile group definitions
    # Extracted from define_tile_groups in OverworldShuffle.py
    # Standard: merge_groups([[0x13, 0x14, 0x1b]])
    # Connectors: [[0x03, 0x0a], [0x28, 0x29]], [[0x05, 0x07]]
    tile_groups = {
        "standard_mode": [0x13, 0x14, 0x1b],
        "connector_caves": [
            [0x03, 0x0a], # Old Man Cave / Bumper Cave
            [0x28, 0x29], # Maze Race / Dig Game (though these are edges, they are linked tiles too)
            [0x05, 0x07]  # EDM / TR
        ]
    }

    # 5. Tile position grid (0-63)
    # Position N has LW tile N and DW tile N+64.
    tile_grid = {}
    for i in range(64):
        tile_grid[i] = {"lw": i, "dw": i + 0x40}

    # 6. Non-parallel tiles
    # Tiles that don't have a direct counterpart in parallel_links or are specifically excluded.
    # From define_tile_groups: [[0x00, 0x2d, 0x80], [0x0f, 0x81], [0x1a, 0x1b], [0x28, 0x29], [0x30, 0x3a]]
    # These contain non-parallel IDs like 0x80, 0x81, etc.
    non_parallel_tiles = [0x00, 0x0f, 0x1a, 0x2d, 0x30, 0x3a, 0x80, 0x81] # MS Pedestal, Zoras Domain, etc.

    # Generate TypeScript output

    # Just use logic_regions for this
    # export const fluteSpotRegistry = {json.dumps(flute_spots, indent=2)};
    # Simple logic, add in tracker
    # export const tilePositionGrid = {json.dumps(tile_grid, indent=2)};

    ts_content = f"""
export const similarEdgeGroups = {json.dumps(similar_edge_groups, indent=2)};

export const whirlpoolRegistry = {json.dumps(whirlpools, indent=2)};

export const defaultTileGroups = {json.dumps(tile_groups, indent=2)};

export const nonParallelTiles = {json.dumps(non_parallel_tiles, indent=2)};
"""
    
    with open('owData.ts', 'w') as f:
        f.write(ts_content.strip())
    print("Generated owData.ts successfully.")

extract_ow_data()

Generated owData.ts successfully.


In [40]:
# === Static key logic (key_logic_algorithm: static) ===
# Emits src/data/logic/staticKeyLogic.ts: an overlay the tracker applies on top of logic_regions when
# its "static" key logic setting is active. Rules are read straight from OWR rather than re-derived:
#   - door thresholds from world.key_logic door_rules after static_door_rules / TR adjustments
#   - location and entrance rules by recording every add_rule/set_rule/forbid_item that
#     set_static_key_rules makes
# Generation is stopped right after set_static_key_rules (before fill), so forcing each Turtle Rock
# variant cannot keylock. Placement conditions ("big key in X", "small key at Y") become
# assume|bigKey|<id>|<locs> / assume|smallKey|<id>|<locs> tokens, which the tracker always treats as
# "possible" with a reason; it never resolves them against checks or item data.
import json as _json

import BaseClasses
import Rules
import source.logic.AccessRule as AR
import source.dungeon.StaticKeyLogic as SKL
from BaseClasses import KeyRuleType
from Dungeons import dungeon_keys, dungeon_bigs

STATIC_DUNGEON_IDS = {
    'Hyrule Castle': 'hc', 'Eastern Palace': 'ep', 'Desert Palace': 'dp', 'Tower of Hera': 'toh',
    'Agahnims Tower': 'ct', 'Palace of Darkness': 'pod', 'Swamp Palace': 'sp', 'Skull Woods': 'sw',
    'Thieves Town': 'tt', 'Ice Palace': 'ip', 'Misery Mire': 'mm', 'Turtle Rock': 'tr', 'Ganons Tower': 'gt',
}
_KEY_TO_ID = {v: STATIC_DUNGEON_IDS[k] for k, v in dungeon_keys.items()}
_BIG_TO_ID = {v: STATIC_DUNGEON_IDS[k] for k, v in dungeon_bigs.items()}
_STATIC_ITEMS = {'Hammer': 'hammer', 'Cane of Somaria': 'somaria'}

STATIC_TR_PROBE = {'back': 'TR Eye Bridge', 'front': 'TR Main Lobby', 'middle': 'TR Lazy Eyes',
                   'bigChest': 'TR Big Chest Entrance'}
# Region reachability that makes set_turtle_rock_rules take each branch (if back / elif front and
# middle / elif front / elif big_chest / otherwise the plain table)
STATIC_TR_VARIANT_FLAGS = {
    'back':        {'back': True,  'front': True,  'middle': True,  'bigChest': True},
    'frontMiddle': {'back': False, 'front': True,  'middle': True,  'bigChest': True},
    'front':       {'back': False, 'front': True,  'middle': False, 'bigChest': True},
    'bigChest':    {'back': False, 'front': False, 'middle': False, 'bigChest': True},
    'none':        {'back': False, 'front': False, 'middle': False, 'bigChest': False},
}


def _static_assume(item, locations):
    if item in _BIG_TO_ID:
        return f"assume|bigKey|{_BIG_TO_ID[item]}|{';'.join(locations)}"
    if item in _KEY_TO_ID:
        return f"assume|smallKey|{_KEY_TO_ID[item]}|{';'.join(locations)}"
    raise ValueError(f'unexpected placement item {item}')


def _static_simplify(node):
    if isinstance(node, dict) and len(node) == 1:
        (key, parts), = node.items()
        flat = []
        for p in (_static_simplify(x) for x in parts):
            for q in (p[key] if isinstance(p, dict) and set(p) == {key} else [p]):
                if q not in flat:
                    flat.append(q)
        if key == 'allOf':
            flat = [p for p in flat if p != {}]
            if 'never' in flat:
                return 'never'
            if not flat:
                return {}
        else:
            if any(p == {} for p in flat):
                return {}
            flat = [p for p in flat if p != 'never']
            if not flat:
                return 'never'
        return flat[0] if len(flat) == 1 else {key: flat}
    return node


def _static_render(rule):
    """AccessRule tree -> tracker LogicRequirement ({} true, 'never' false)."""
    if isinstance(rule, AR.TrueRule):
        return {}
    if isinstance(rule, AR.FalseRule):
        return 'never'
    if isinstance(rule, AR.Has):
        if rule.item in _KEY_TO_ID:
            return f'keys|{_KEY_TO_ID[rule.item]}|{rule.count}'
        return _STATIC_ITEMS[rule.item]
    if isinstance(rule, AR.Primitive):
        name, args_ = rule.method_name, rule.args
        if name == 'has_sm_key_strict':
            return f'keys|{_KEY_TO_ID[args_[0]]}|{args_[2] if len(args_) > 2 else 1}'
        if name == 'item_in_named_locations':
            return _static_assume(args_[0], list(args_[2]))
        if name == 'item_is_at':
            return _static_assume(args_[2], [args_[0]])
        raise ValueError(f'unhandled primitive {name}{args_}')
    if isinstance(rule, AR.NotRule):
        # Only TR keys_for_chest_item uses not_rule, to make its last branch exclusive:
        # or(A, B & k2, !A & !B & k4) == or(A, B & k2, k4) because k4 implies k2.
        inner = _static_render(rule.rule)
        assert isinstance(inner, str) and inner.startswith('assume|'), f'unexpected not_rule({inner})'
        return {}
    if isinstance(rule, (AR.AndRule, AR.OrRule)):
        key = 'allOf' if isinstance(rule, AR.AndRule) else 'anyOf'
        return _static_simplify({key: [_static_render(r) for r in rule.rules]})
    raise ValueError(f'unhandled rule node {type(rule).__name__}: {rule}')


def _static_door_requirement(rule, dungeon):
    did = STATIC_DUNGEON_IDS[dungeon]
    alts = []
    for rule_type, number in rule.new_rules.items():
        if rule_type == KeyRuleType.WorstCase:
            alts.append(f'keys|{did}|{number}')
        elif rule_type == KeyRuleType.AllowSmall:
            alts.append({'allOf': [f'assume|smallKey|{did}|{rule.small_location.name}', f'keys|{did}|{number}']})
        elif isinstance(rule_type, tuple) and rule_type[0] == KeyRuleType.Lock:
            locs = sorted(l.name if hasattr(l, 'name') else l for l in rule.alternate_big_key_loc)
            alts.append({'allOf': [f"assume|bigKey|{did}|{';'.join(locs)}", f'keys|{did}|{number}']})
        else:
            raise ValueError(f'unhandled door rule type {rule_type}')
    return _static_simplify({'anyOf': alts})


class _StopAfterStatic(Exception):
    def __init__(self, world, records):
        super().__init__('stopped after set_static_key_rules')
        self.world, self.records = world, records


def capture_static_world(mode, tr_variant):
    """Run main() until set_static_key_rules returns, recording every rule it adds."""
    records = []
    orig_static, orig_tr = SKL.set_static_key_rules, SKL.set_turtle_rock_rules
    orig_add, orig_set = AR.add_rule, AR.set_rule
    orig_forbid, orig_allow = Rules.forbid_item, Rules.set_always_allow
    orig_can_reach = BaseClasses.Region.can_reach

    def kind(spot):
        return 'location' if isinstance(spot, BaseClasses.Location) else 'exit'

    def add_rule(spot, rule, combine='and'):
        records.append(('or' if combine == 'or' else 'and', kind(spot), spot.name, rule))
        return orig_add(spot, rule, combine)

    def set_rule(spot, rule):
        records.append(('replace', kind(spot), spot.name, rule))
        return orig_set(spot, rule)

    def forbid_item(location, item, player):
        records.append(('forbid', 'location', location.name, item))
        return orig_forbid(location, item, player)

    def tr_rules(world, *a, **kw):
        forced = {STATIC_TR_PROBE[k]: v for k, v in STATIC_TR_VARIANT_FLAGS[tr_variant].items()}

        def can_reach(region, state):
            return forced[region.name] if region.name in forced else orig_can_reach(region, state)
        BaseClasses.Region.can_reach = can_reach
        try:
            return orig_tr(world, *a, **kw)
        finally:
            BaseClasses.Region.can_reach = orig_can_reach

    def static_rules(world, player):
        AR.add_rule, AR.set_rule, Rules.forbid_item = add_rule, set_rule, forbid_item
        try:
            orig_static(world, player)
        finally:
            AR.add_rule, AR.set_rule, Rules.forbid_item = orig_add, orig_set, orig_forbid
        raise _StopAfterStatic(world, records)

    keys_ = ('key_logic_algorithm', 'pottery', 'dropshuffle', 'mode', 'logic', 'door_shuffle')
    saved = {k: getattr(args, k)[1] for k in keys_}
    args.key_logic_algorithm[1], args.pottery[1], args.dropshuffle[1] = 'static', 'none', 'none'
    args.mode[1], args.logic[1], args.door_shuffle[1] = mode, 'noglitches', 'vanilla'
    SKL.set_static_key_rules, SKL.set_turtle_rock_rules = static_rules, tr_rules
    try:
        main(args=args, fish=fish, seed=0)
    except _StopAfterStatic as stop:
        return stop.world, stop.records
    finally:
        SKL.set_static_key_rules, SKL.set_turtle_rock_rules = orig_static, orig_tr
        Rules.set_always_allow = orig_allow
        for k, v in saved.items():
            getattr(args, k)[1] = v
    raise RuntimeError('set_static_key_rules was never called')


def extract_static_world(world, records):
    player = 1
    doors, locations, entrance_or, forbidden = {}, {}, {}, {}
    for dungeon, key_logic in world.key_logic[player].items():
        for door_name, rule in key_logic.door_rules.items():
            if KeyRuleType.WorstCase not in rule.new_rules:
                continue
            door = world.get_door(door_name, player)
            doors[door_name] = {'region': door.entrance.parent_region.name, 'dungeon': STATIC_DUNGEON_IDS[dungeon],
                                'rule': _static_door_requirement(rule, dungeon)}
    for op, kind, name, rule in records:
        if op == 'forbid':
            forbidden.setdefault(name, set()).add(rule)
        elif kind == 'exit':
            assert op == 'or', f'unexpected static entrance rule {op} on {name}'
            entrance_or.setdefault(name, []).append(_static_render(rule))
        else:
            loc = world.get_location(name, player)
            prev = locations.get(name)
            req = _static_render(rule)
            if prev and op == 'and':
                req = _static_simplify({'allOf': [prev['rule'], req]})
            locations[name] = {'region': loc.parent_region.name,
                               'dungeon': STATIC_DUNGEON_IDS[loc.parent_region.dungeon.name],
                               'op': 'replace' if op == 'replace' or (prev and prev['op'] == 'replace') else 'and',
                               'rule': req}
    for name, extras in entrance_or.items():
        assert name in doors, f'static OR rule on a non key door: {name}'
        doors[name]['rule'] = _static_simplify({'anyOf': [doors[name]['rule']] + extras})
    candidates = {}
    for dungeon in world.dungeons:
        if dungeon.player != player or dungeon.name not in STATIC_DUNGEON_IDS:
            continue
        key = dungeon_keys[dungeon.name]
        names = []
        for region in dungeon.regions:
            region = world.get_region(region, player) if isinstance(region, str) else region
            for loc in region.locations:
                if loc.type == 0 and not loc.event and key not in forbidden.get(loc.name, ()):
                    names.append(loc.name)
        candidates[STATIC_DUNGEON_IDS[dungeon.name]] = sorted(names)
    return doors, locations, candidates


def build_static_key_logic():
    by_variant = {}
    for variant in STATIC_TR_VARIANT_FLAGS:
        by_variant[variant] = extract_static_world(*capture_static_world('open', variant))
    std_doors, std_locations, std_candidates = extract_static_world(*capture_static_world('standard', 'back'))
    open_doors, open_locations, open_candidates = by_variant['back']

    def merge(entries_by_variant, name):
        rules = {v: e[name]['rule'] if name in e else {} for v, e in entries_by_variant.items()}
        if len({_json.dumps(r, sort_keys=True) for r in rules.values()}) == 1:
            return rules['back']
        return {'tr': rules}

    doors = {}
    for name in sorted(set().union(*(set(v[0]) for v in by_variant.values()))):
        base = next(v[0][name] for v in by_variant.values() if name in v[0])
        doors[name] = {'region': base['region'], 'dungeon': base['dungeon'],
                       'rule': merge({k: v[0] for k, v in by_variant.items()}, name)}
        assert _json.dumps(std_doors.get(name, {}).get('rule'), sort_keys=True) == \
            _json.dumps(open_doors.get(name, {}).get('rule'), sort_keys=True), f'standard door rule differs: {name}'
    locations = {}
    for name in sorted(set().union(*(set(v[1]) for v in by_variant.values()))):
        base = next(v[1][name] for v in by_variant.values() if name in v[1])
        entry = {'region': base['region'], 'dungeon': base['dungeon'], 'op': base['op'],
                 'rule': merge({k: v[1] for k, v in by_variant.items()}, name)}
        if name not in std_locations:
            entry['modes'] = ['nonstandard']
        locations[name] = entry
    candidates = {did: {'all': std_candidates[did],
                        'nonStandardExclude': sorted(set(std_candidates[did]) - set(open_candidates[did]))}
                  for did in std_candidates}
    tr_key_doors = sorted(n for n, d in doors.items() if d['dungeon'] == 'tr')
    return doors, locations, candidates, dict(STATIC_TR_PROBE, keyDoorExits=tr_key_doors)


def write_static_key_logic(paths=('staticKeyLogic.ts', '../alttptracker_react/src/data/logic/staticKeyLogic.ts')):
    doors, locations, candidates, probe = build_static_key_logic()
    body = (
        '// This file is auto-generated by extractRegionsGraph.ipynb (static key logic cell)\n'
        'import type { StaticDoorRule, StaticLocationRule, StaticKeyCandidates, StaticTRProbe } '
        'from "@/data/logic/logicTypes";\n\n'
        f'export const STATIC_DOOR_RULES: Record<string, StaticDoorRule> = {_json.dumps(doors, indent=4)};\n\n'
        f'export const STATIC_LOCATION_RULES: Record<string, StaticLocationRule> = {_json.dumps(locations, indent=4)};\n\n'
        f'export const STATIC_KEY_CANDIDATES: Record<string, StaticKeyCandidates> = {_json.dumps(candidates, indent=4)};\n\n'
        f'export const STATIC_TR_PROBE: StaticTRProbe = {_json.dumps(probe, indent=4)};\n'
    )
    for path in paths:
        with open(path, 'w') as f:
            f.write(body)
    print(f'Static key logic: {len(doors)} doors, {len(locations)} location rules, '
          f'{sum(len(c["all"]) for c in candidates.values())} key candidates.')


write_static_key_logic()

Static key logic: 110 doors, 14 location rules, 117 key candidates.


In [ ]:
! cp /mnt/dev/ALttPOverworldRandomizer/extractRegionsGraph.ipynb /mnt/dev/alttptracker_react/